# load files

## Measurement data

### summarizing all of the non-sequencing data in `measurements/*.csv`
- VFA.csv: Contains Acetate, Propionate, Butyrate, and COD for the reactor, waste, and media
- Ammonia.csv: Ammonia concentrations in the reactor, waste, and media
- TSS_VSS.csv: Abundances of TSS & VSS in the system
- Pre-Summary.csv: Defining sampling codes as days of operation
- GC_TCD.csv: gaseous components of various reactors
- Summary.csv: `main content`.  Complete gaseous ins/outs, carbon accounting, occasional media VFA and TSS measurements

In [ ]:
from pandas import read_excel

measurements = read_excel("model_inputs/measurements.xlsx", engine='openpyxl', sheet_name=None)
print(len(measurements))
for sheetName, csv in measurements.items():
    csv.to_csv(f"model_inputs/{sheetname}.csv")

### Process Summary.csv

In [ ]:
# --- MIGRATED: operational bridge already rebuilt for the 16 mature samples ---
from json import load
summary_samples = load(open("model_inputs/measurements/summary_samples.json", "r"))
print(f"summary_samples.json: {len(summary_samples)} mature samples")


### ModelSEED compounds for these column names

In [ ]:
summary_key_to_modelseed = {

    # ---- MEDIA COMPOUNDS ----
    "Media Acetate (mol/L)": "cpd00029",        # acetate
    "Media Propionate (mol/L)": "cpd00141",     # propionate
    "Media Butyrate (mol/L)": "cpd00211",       # butyrate
    "Media Ammonia (mol/L)": "cpd00013",        # NH3/NH4+
    "Media Dissolved Inorganic Carbon (mol C/L)": "cpd00011",  # CO2 (aq)
    
    # ---- GAS FEED ----
    "H2 delivery rate (mol/min)": "cpd00067",   # hydrogen
    "CO2 delivery rate (mol/min)": "cpd00011",  # carbon dioxide

    # ---- WASTE / EFFLUENT COMPOUNDS ----
    "Waste Effluent Acetate (mol/L)": "cpd00029",
    "Waste Effluent Propionate (mol/L)": "cpd00141",
    "Waste Effluent Butyrate (mol/L)": "cpd00211",
    "Waste Effluent Ammonia (mol/L)": "cpd00013",
    "Waste Effluent Dissolved Inorganic Carbon (mol C/L)": "cpd00011",
    "Waste Effluent Dissolved Organic Carbon (mol C/L)": "cpd00027",  # generic glucose-equivalent DOC proxy

    # ---- GAS PHASE ----
    "Gas Composition (% CH4)": "cpd00104",      # methane
    "Gas Composition (% H2)": "cpd00067",
    "Gas Composition (% CO2)": "cpd00011",
    "CH4 Production (mol/min)": "cpd00104",
    "H2 breakthrough (mol/min)": "cpd00067",
    "CO2 breakthrough (mol/min)": "cpd00011",

    # ---- AQUEOUS CARBONATE SPECIATION ----
    "[CO2aq]": "cpd00011",
    "[HCO3-]": "cpd00099",   # bicarbonate
    "[CO3-2]": "cpd00060",  # carbonate

}


## Taxonomy and Abundance data

In [ ]:
# --- MIGRATED: load the new_data foundation (all 3,276 ASVs labelled; abundance
# table scoped to the 16 mature samples and the ASVs present in them) ---
from json import load, dump
from pandas import DataFrame
import numpy as np
ab_js = load(open("model_inputs/abundances.json", "r"))          # {sample: {seq: fraction}}  16 mature samples
full_taxonomy = load(open("model_inputs/taxonomy.json", "r"))    # {seq: {Kingdom..Species}}  all 3,276
present = set(next(iter(ab_js.values())).keys())                 # ASVs present in the mature community
taxonomy_js = {seq: full_taxonomy[seq] for seq in present}       # scope taxonomic charts to the mature community
meaningful_ASVs = DataFrame(ab_js)                               # index=seq, cols=mature samples
print(f"abundances: {meaningful_ASVs.shape[0]} present ASVs x {meaningful_ASVs.shape[1]} mature samples; "
      f"taxonomy(all): {len(full_taxonomy)} ASVs")


In [ ]:
# from json import load, dump
# IDmd5 = load(open("model_inputs/ID_md5.json", 'r'))
# dump({v: k.split()[1].split(".rna")[0] for k,v in IDmd5.items()}, open("model_inputs/md5_ID.json", 'w'), indent=2)
# list(IDmd5.items())[:2]

# Taxonomy

### Create iterative IDs

In [ ]:
# --- MIGRATED: load iterativeIDs (261 published labels preserved + extended to all 3,276) ---
from json import load
iterativeIDs = load(open("model_inputs/iterativeIDs.json", "r"))            # {seq: ID}  all 3,276
iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", "r"))  # {ID: rank}
print(f"iterativeIDs: {len(iterativeIDs)} ASVs labelled")


## taxonomic representation

In [ ]:
import matplotlib.pyplot as plt
from numpy import nan

def taxa_bar_chart(taxa_asvs, x_label, title, export_name):
        # Create the bar chart
        limit_size = 40
        sorted_data = dict(sorted({k:len(x) for k,x in taxa_asvs.items()}.items(), key=lambda x: x[1], reverse=True))
        sorted_data = dict(list(sorted_data.items())[:limit_size])
        plt.bar(list(sorted_data.keys()), list(sorted_data.values()), color='lightblue', edgecolor='black')

        # Add labels on top of each bar
        for i, value in enumerate(list(sorted_data.values())):
            plt.text(i, value + max(list(sorted_data.values()))/30, str(value), ha='center', fontsize=12, color='black', rotation=90)

        # Add labels and title
        plt.xlabel(x_label)
        plt.ylabel(f"Frequency ({sum(list(sorted_data.values()))} total)")
        plt.title(title)

        # Rotate x-axis labels if needed
        plt.xticks(rotation=90)

        # Show the plot
        plt.tight_layout()
        plt.savefig(export_name)
        # plt.show()


def define_taxa_asvs(taxa, full_taxonomy):
    taxa_asvs = {}
    for asv, taxonomy in full_taxonomy.items():
        if str(taxonomy.get(taxa, 'none')) in [nan, "nan", "NaN", "none", "None"] or "midas" in taxonomy.get(taxa, 'none'):
            continue
        if taxonomy.get(taxa, 'none') not in taxa_asvs:
            taxa_asvs[taxonomy.get(taxa, 'none')] = []
        taxa_asvs[taxonomy.get(taxa, 'none')].append(asv)
    return taxa_asvs


for taxa in ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]:
    taxa_asvs = define_taxa_asvs(taxa, taxonomy_js)
    taxa_bar_chart(taxa_asvs, taxa,
        f"{taxa} ({len(taxa_asvs)} total) ASVs", f"taxonomic_representations/{taxa}_asvs.png")


# from deepdiff import DeepDiff
for taxa in ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]:
    for sample, content in ab_js.items():
        full_taxonomy = {k:taxonomy_js[k] for k, v in content.items() if v > 0}
        # display(DeepDiff(taxonomy_js, full_taxonomy))
        taxa_asvs = define_taxa_asvs(taxa, full_taxonomy)
        taxa_bar_chart(taxa_asvs, taxa,
            f"{sample} {taxa} ({len(taxa_asvs)} total) ASVs", f"taxonomic_representations/{sample}_{taxa}_asvs.png")
        # break

# Abundance

In [ ]:
from json import dump
from matplotlib import pyplot as plt
from numpy import array 
for sample, content in ab_js.items(): 
    print(sample)
    abundances = array([x for x in list(content.values()) if x > 0])
    # print(len(abundances))

    plt.hist(abundances, bins=200)

    # Add labels and title
    plt.xlabel('% Relative Abundance')
    plt.ylabel(f'ASVs ({len(abundances)} total)')
    plt.yscale('log')
    plt.title(f'{sample} ASV Abundance')

    # Display the plot
    plt.savefig(f'abundance_distributions/{sample}_abundance.png')
    plt.show()

## Shannon Diversity Index

In [ ]:
import matplotlib.pyplot as plt
from numpy import array, log
from json import load

asvSet_abundances = load(open(f"modeling_files/ASVset_abundances.json", 'r'))

SDI = {}
for sample, asvAbund in asvSet_abundances.items():
    abundances = array(list(asvAbund.values()))
    abundances /= sum(abundances)
    SDI[sample] = -sum(log(abundances)*abundances)

plt.figure(figsize=(10, 6))
plt.plot(SDI.keys(), SDI.values(), marker='o', linestyle='-', color='blue', linewidth=2, markersize=8)
plt.xlabel('Samples')
plt.ylabel('Shannon Index', fontsize=20)
plt.title('Shannon Diversity Index for all samples', fontsize=20)
plt.grid(True)
plt.yticks(fontsize=20)  # Rotate labels if needed
plt.xticks(rotation=60, fontsize=20)  # Rotate labels if needed
plt.tight_layout()
plt.show()

## Abundance heatmaps

In [ ]:
def taxonomy_linkage(taxonomy_series):
    """
    Build a linkage matrix that EXACTLY follows taxonomy hierarchy.
    Auto-detects depth from the taxonomy strings.
    """
    # Parse taxonomy - detect separator and split
    parsed = (
        taxonomy_series.fillna("unknown")
        .astype(str)
        .str.split(r"[;|,]", regex=True)
        .apply(lambda lst: [x.strip() for x in (lst or []) if x.strip()])
    )
    
    # Auto-detect max depth from data
    ranks_n = max(len(p) for p in parsed)
    
    if ranks_n == 0:
        ranks_n = 1  # Minimum depth
    
    # Pad to consistent length
    parsed = parsed.apply(lambda lst: (lst + [""] * ranks_n)[:ranks_n])
    
    n = len(parsed)
    if n <= 1:
        return np.empty((0, 4), dtype=float)
    
    # Map original indices to positions 0..n-1
    idx_to_pos = {idx: pos for pos, idx in enumerate(parsed.index)}
    
    linkage_rows = []
    next_cluster_id = n
    
    def build_subtree(indices, depth):
        nonlocal next_cluster_id
        
        # Base case: single item
        if len(indices) == 1:
            return idx_to_pos[indices[0]], 1
        
        # Base case: reached max depth, merge all remaining
        if depth >= ranks_n:
            merge_distance = 0.5  # Small distance for items identical up to max depth
            cluster_id, total_count = idx_to_pos[indices[0]], 1
            for idx in indices[1:]:
                new_id = next_cluster_id
                next_cluster_id += 1
                linkage_rows.append([cluster_id, idx_to_pos[idx], merge_distance, total_count + 1])
                cluster_id = new_id
                total_count += 1
            return cluster_id, total_count
        
        # Group by taxonomy at current depth
        groups = defaultdict(list)
        for idx in indices:
            rank_value = parsed.loc[idx][depth]
            if rank_value == "":
                # Empty ranks get unique keys so they don't group together
                rank_value = f"__unclassified_{idx}"
            groups[rank_value].append(idx)
        
        # Recursively build subtrees for each group
        subclusters = []
        for rank_value, group_indices in groups.items():
            sub_id, sub_count = build_subtree(group_indices, depth + 1)
            subclusters.append((sub_id, sub_count))
        
        # If only one subcluster, no merge needed at this level
        if len(subclusters) == 1:
            return subclusters[0]
        
        # Merge subclusters at this depth
        # Distance increases as we go higher in taxonomy (closer to root = larger distance)
        merge_distance = float(ranks_n - depth)
        
        cluster_id, total_count = subclusters[0]
        for sub_id, sub_count in subclusters[1:]:
            new_id = next_cluster_id
            next_cluster_id += 1
            linkage_rows.append([cluster_id, sub_id, merge_distance, total_count + sub_count])
            cluster_id = new_id
            total_count += sub_count
        
        return cluster_id, total_count
    
    build_subtree(list(parsed.index), depth=0)
    
    Z = array(linkage_rows, dtype=float)
    
    # Validate linkage matrix (distances must be monotonically increasing for scipy)
    # Re-normalize distances to ensure monotonicity
    if len(Z) > 0:
        # Sort by original merge order but ensure distances are non-decreasing
        for i in range(1, len(Z)):
            if Z[i, 2] < Z[i-1, 2]:
                Z[i, 2] = Z[i-1, 2]
    
    return Z


In [ ]:
from pandas import read_csv, DataFrame, Series, concat
import seaborn as sns
from matplotlib import pyplot
from json import load
from numpy import inf, nan, empty, logspace, array
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage
from collections import Counter, defaultdict

# taxonomy = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/digestor/new_abundances.json", 'r'))


def concatenate_taxonomy(df, level):
    dic = {}
    for index, row in df.iterrows():
        dic["|".join([str(row[l]) for l in taxonomic_levels
                    if taxonomic_levels.index(l) <= taxonomic_levels.index(level)])] = index
    return Series(dic)

def change_columns(df):
    return df[[c for c in sample_days.values() if c in df.columns]]
    # return df[[c for c in sample_days if c in df.columns]]


from matplotlib.colors import LinearSegmentedColormap

# ab = read_csv("GAME Sequencing/ASV_abundance_wideform.csv").set_index("seq")
taxonomic_levels = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
sample_days = {
    "5-10": 0,
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}

total_df = read_csv("model_inputs/total.csv").set_index("seq")
max_abundance = total_df["rel_ab"].max()
print("max", max_abundance)
# sample_dfs = []
# for sample in sample_days:
#     sample_og = og[og["sample"] == sample]
#     sample_og["rel_ab"] = sample_og["rel_ab"] / sample_og["rel_ab"].sum()
#     sample_dfs.append(sample_og)
# og = concat(sample_dfs)
total_df["rel_ab"] /= 100
max_abundance = total_df["rel_ab"].max()
print("max", max_abundance)
intervals = [0] + list(logspace(-2, -.3, 5))
print(intervals)


def create_heatmap(df, level, abundance_range, taxonomies, cmap=None, log=False):
    # min_abundance = min(list(df.to_numpy().flatten()))
    # max_abundance = max(list(df.to_numpy().flatten()))
    # new_cmap = LinearSegmentedColormap.from_list("NewMap", [(0, "skyblue"), (1, "red")])
    # if log:
    cmap = cmap or LinearSegmentedColormap.from_list("NewMap", ["skyblue", "red"])
    # display(df)
    # clustermap = ClusterHeatmap(df, level, ) # Series(dict(Counter(taxonomies))))
    clusterMap = sns.clustermap(df,
                                # row_colors=row_colors,
                                cmap=cmap,
                                vmin=0,
                                vmax=max(list(df.to_numpy().flatten())),
                                col_cluster=False, figsize=(20, 20),
                                row_linkage=taxonomy_linkage(Series(taxonomies)),
                                dendrogram_ratio=(.1, .2))
    clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
    clusterMap.figure.suptitle(f"{level.capitalize()} {abundance_range} Abundances", fontsize=30)
    clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=20, rotation=0)
    clusterMap.ax_heatmap.set_xticklabels(clusterMap.ax_heatmap.get_xticklabels(), fontsize=24)
    clusterMap.ax_heatmap.set_xlabel("Days after inoculation", fontsize=30)
    clusterMap.figure.tight_layout()
    clusterMap.figure.savefig(f"abundance_heatmaps/{level}_{abundance_range}_abundances.png")

# taxa- and abundance-level heatmaps
for i, abundance_limit in enumerate(intervals):
    if abundance_limit == 0:  continue
    previous_limit = intervals[i-1]
    print(previous_limit, abundance_limit)
    abundance_range = f"{round(previous_limit, 2)}-{round(abundance_limit, 2)}"
    for level in taxonomic_levels:
        # ab = ab.set_index(level).drop([l for l in taxonomic_levels if l != level], axis=1)#.groupby(level).sum()
        # display(ab)

        dic = {}
        taxonomies = {}
        for seq, row in total_df.iterrows():
            # for s, abundance in row.items():
            day = sample_days.get(row["sample"])
            if day is None:  continue
            # print(day)
            
            # print(abundance)
            abundance = row["rel_ab"]
            if abundance <= previous_limit or abundance > abundance_limit:  continue

            taxonomies.setdefault(row[level], "|".join([str(row[l]) for l in taxonomic_levels
                                    if taxonomic_levels.index(l) <= taxonomic_levels.index(level)]))
            dic.setdefault(day, {})
            dic[day].setdefault(row[level], 0)
            dic[day][row[level]] += abundance

        # display(dic)
        if dic == {}:
            print(f"The {level} does not have organisms in {abundance_range}")
            continue
        elif len(taxonomies) == 1:
            print(f"The {level} only has one organism in {abundance_range}:  {dic} and {taxonomies}")
            continue

        if level == "Phylum":  display(taxonomies)

        # display(dic)

        df = change_columns(DataFrame(dic))
        df = df.astype(float).replace([inf, -inf], nan).fillna(0)
        # tax_series = Series(taxonomies)
        # print("Taxonomy index matches df.index:", tax_series.index.equals(df.index))
        # print("Taxonomy index order matches:", list(tax_series.index) == list(df.index))

        # display(df)
        # display(df.to_numpy())
        # Create series from df.index
        taxonomy_series = Series({idx: taxonomies.get(idx, f"Unknown|{idx}") for idx in df.index})

        # Debug: check detected depth
        print(f"Detected depth: {max(len(t.split('|')) for t in taxonomy_series)}")
        print(f"Sample entries:\n{taxonomy_series.head()}")
        # max_abundance = max(list(df.to_numpy().flatten()))
        create_heatmap(df, level, abundance_range, taxonomy_series)
        # clustermap = ClusterHeatmap(df, level, ) # Series(dict(Counter(taxonomies))))
        # clusterMap = sns.clustermap(df, 
        #                             # row_colors=row_colors,
        #                             cmap=new_cmap,
        #                             vmin=0,
        #                             vmax=max_abundance,
        #                             col_cluster=False, figsize=(20, 20),
        #                             row_linkage=taxonomy_linkage(Series(taxonomies)),
        #                             dendrogram_ratio=(.1, .2))
        # clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
        # clusterMap.figure.suptitle(f"{level.capitalize()} {abundance_range} Abundances", fontsize=30)
        # clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=20, rotation=0)
        # clusterMap.ax_heatmap.set_xticklabels(clusterMap.ax_heatmap.get_xticklabels(), fontsize=24)
        # clusterMap.ax_heatmap.set_xlabel("Days after inoculation", fontsize=30)
        # clusterMap.figure.tight_layout()
        # clusterMap.figure.savefig(f"abundance_heatmaps/{level}_{abundance_range}_abundances.png")

In [ ]:
taxonomies

#### Making Heather's requested figures

In [ ]:
from json import load as _ldj
taxonomy = _ldj(open("model_inputs/taxonomy.json"))
from json import load, dump                                                                                                                                                 
from pathlib import Path                                        
import matplotlib.pyplot as plt

def rgba_to_hex(rgba):                                                                                                                                                      
    return "#{:02x}{:02x}{:02x}".format(*(int(c*255) for c in rgba[:3]))
                                                                                                                                                                            
def build_phylum_colors(out_path="model_inputs/phylum_colors.json"):                                                                                                           
    phyla = sorted({t.get("Phylum") for t in taxonomy.values() if t.get("Phylum")})                                                                                         

    archaea_markers = ["archaeo", "euryarchaeota", "crenarchaeota", "thaumarchaeota",                                                                                       
                        "halobacterota", "methanobacteriota", "micrarchaeota", "nanoarchaeota"]
    archaea  = [p for p in phyla if any(m in p.lower() for m in archaea_markers)]                                                                                           
    bacteria = [p for p in phyla if p not in archaea]           
                                                                                                                                                                            
    colors = {}                                                 
    for i, p in enumerate(archaea):                                                                                                                                         
        colors[p] = rgba_to_hex(plt.cm.turbo(i / max(len(archaea), 1) * 0.15))
    for i, p in enumerate(bacteria):                                                                                                                                        
        colors[p] = rgba_to_hex(plt.cm.turbo(0.2 + i / max(len(bacteria), 1) * 0.8))
                                                                                                                                                                            
    dump(colors, open(out_path, "w"), indent=2)                 
    return colors                                                                                                                                                           
                                                                
build_phylum_colors()

# In every figure script, load instead of rebuilding:                                                                                                                         

from json import load                                                                                                                                                       
phylum_colors = load(open("model_inputs/phylum_colors.json", 'r'))                                                                                                               
iterativeID_color_map = load(open(f"iterativeID_color_map.json", "r"))
display(iterativeID_color_map) 
DEFAULT = "lightgray"                                                                                                                                                       

# Then consume it wherever you currently build colors ad-hoc:                                                                                                                 
                                                                
# network                                                                                                                                                                   
node_colors = [phylum_colors.get(iterativeID_level.get(n, "Unknown"), DEFAULT) for n in G.nodes()]                                                                          

# heatmap row_colors          
print(df.index)
if ("." in df.index[0]):
    row_colors = [iterativeID_color_map[org] for org in df.index]
else:
    genera_color_map = {ID.split(".")[0]: v for ID, v in iterativeID_color_map.items()}
    row_colors = [genera_color_map.get(org, DEFAULT) for org in df.index] #.split(".")[0]
# print(iterativeID_color_map[org])
print(len(row_colors), row_colors)                                                                                                                                                              
# legend patches in any figure                                                                                                                                              
archaea_patches  = [mpatches.Patch(color=phylum_colors[p], label=p) for p in archaea_phyla]
bacteria_patches = [mpatches.Patch(color=phylum_colors[p], label=p) for p in bacteria_phyla]

In [ ]:
import numpy as np
from matplotlib.colors import LogNorm, Normalize, BoundaryNorm, TwoSlopeNorm
from matplotlib.patches import Patch  
from matplotlib.transforms import Bbox
from scipy.cluster import hierarchy
from pandas import read_csv, DataFrame, Series
import sigfig
import seaborn as sns
from numpy import log10, delete, nanmean, where, isnan, logspace, inf, nan, array
from json import load
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter, defaultdict


# ab = read_csv("GAME Sequencing/ASV_abundance_wideform.csv").set_index("seq")
taxonomic_levels = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
sample_days = {
    "5-10": 0,
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}

iterativeID_color_map = load(open(f"iterativeID_color_map.json", "r"))
genera_color_map = {ID.split(".")[0]: v for ID, v in iterativeID_color_map.items()}
total_df = read_csv("model_inputs/total.csv").set_index("seq")
max_abundance = total_df["rel_ab"].max()
print("max", max_abundance)
# sample_dfs = []
# for sample in sample_days:
#     sample_og = og[og["sample"] == sample]
#     sample_og["rel_ab"] = sample_og["rel_ab"] / sample_og["rel_ab"].sum()
#     sample_dfs.append(sample_og)
# og = concat(sample_dfs)
total_df["rel_ab"] /= 100
max_abundance = total_df["rel_ab"].max()
print("max", max_abundance)
intervals = [0] + list(logspace(-2, -.3, 5))
print(intervals)

def create_heatmap(df, taxonomies, title, inlayed_data=None, linkage=None):
    # new_cmap = LinearSegmentedColormap.from_list("NewMap", ["white", "lightblue", "blue"])
    # new_cmap = LinearSegmentedColormap.from_list("NewMap", ["aliceblue", "lightblue", "navy"])
    new_cmap = LinearSegmentedColormap.from_list("NewMap", [        
        (0.00, "aliceblue"),
        (0.25, "lightblue"),   # midpoint color now at 25% of cmap, not 50%
        (1.00, "navy"),                                                                                                                                                         
    ]) 
    new_cmap.set_bad("aliceblue") 
    # display(df)
    # clustermap = ClusterHeatmap(df, level, ) # Series(dict(Counter(taxonomies))))
    # norm = BoundaryNorm([0.01, 0.1], ncolors=new_cmap.N, clip=True)
    vmin = df.min().min()                                                                                                                                                       
    vmax = df.max().max()                                           
    vcenter = log10(0.1)  # 4% in log-fraction space
    # clamp so vmin < vcenter < vmax (TwoSlopeNorm requires strict ordering)                                                                                                    
    vcenter = min(max(vcenter, vmin + 1e-6), vmax - 1e-6)                                                                                                                       
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

    #NOTE There needs to be different row_colors for the different figures, hence the previous cell cannot be used
    DEFAULT_COLOR  = "lightgray"                                                                                                                                            
    def lookup(idx):                                                                                                                                                        
        if idx in iterativeID_color_map:                      # exact (.N suffix)
            return iterativeID_color_map[idx]  
        elif idx in genera_color_map:
            return genera_color_map[idx]
        # stripped = str(idx).split(".")[0]              # try stripped form
        # # find any key whose stripped form matches                                                                                                                          
        # for k, v in taxa_color_map.items():                     
        #     if k.split(".")[0] == stripped:                                                                                                                                 
        #         return v                                        
        return DEFAULT_COLOR                                                                                                                                                
    row_colors = Series({idx: lookup(idx) for idx in df.index}, name="Phylum") 

    clusterMap = sns.clustermap(df,
                                row_colors=row_colors,
                                # cmap="coolwarm_r",
                                cmap=new_cmap,
                                # cbar_kws={"label": "Relative Abundance"},
                                # vmin,
                                # vmax=-.2,
                                norm=norm,
                                clip_on=True,
                                col_cluster=False,
                                row_cluster=True, #False,
                                figsize=(20, 20),
                                # row_linkage=linkage,
                                row_linkage=taxonomy_linkage(taxonomies),
                                dendrogram_ratio=(.2, .15))

    for tick in clusterMap.ax_row_colors.get_xticklabels():                                                                                                                     
        tick.set_fontsize(22)                                                                                                                                                   
        # tick.set_fontweight("bold")     # optional
    # Now customize the colorbar to show ORIGINAL (non-log) values
    cbar = clusterMap.ax_cbar
    # log_ticks = cbar.get_yticks()
    log_ticks = np.array([vmin, log10(0.01), vcenter, log10(0.20), vmax])                                                                                      
    cbar.set_yticks(log_ticks)
    # if "Methanogens" not in title:
    #     log_ticks = log_ticks[::2]
    # else:
    #     log_ticks = delete(log_ticks, 1)
    original_ticks = 10**log_ticks  # or just 10**log_ticks if no +pseudocount
    original_labels = [f"{x*100:.1e}".replace("e-0", "E-") if x*100 < 0.1 else round(x*100) for x in original_ticks]
    original_labels[0] = "0"
    # display(df)
    # print(log_ticks, original_ticks, original_labels)
    cbar.set_yticklabels(original_labels, fontsize=28)
    cbar.set_yticks(log_ticks)  # already there, but ensure
    cbar.set_xlabel("")
    cbar.set_ylabel("Rel. Abundance %", fontsize=32, labelpad=20, rotation=90)
    cbar.tick_params(labelsize=28, length=10, width=2)

    if inlayed_data:
        top_ax = clusterMap.ax_col_dendrogram
        top_ax.clear()
        # Map Shannon-dict day-keys to heatmap column-center positions so the line aligns with each column
        _day_to_idx = {day: i + 0.5 for i, day in enumerate(df.columns)}
        _xs = [_day_to_idx[d] for d in inlayed_data if d in _day_to_idx]
        _ys = [inlayed_data[d] for d in inlayed_data if d in _day_to_idx]
        top_ax.plot(_xs, _ys, color='black', marker='o', linewidth=3)
        top_ax.set_xlim(0, len(df.columns))   # match heatmap x-extent
        top_ax.set_ylabel("Shannon Diversity", fontsize=20)  # or whatever your line represents
        top_ax.yaxis.tick_right()           # moves the tick marks to the right
        top_ax.yaxis.set_label_position("right")
        top_ax.grid(True, axis='y', linestyle='--', alpha=0.7)
        top_ax.set_xticks([])                 # hide x-ticks since they're below on heatmap
        top_ax.tick_params(axis='y', labelsize=16)

    
    ## add values text to the elements
    for i in range(clusterMap.data2d.shape[0]):                                                                                                                                 
      for j in range(clusterMap.data2d.shape[1]):                                                                                                                             
          cell = clusterMap.data2d.iloc[i, j]
          if isnan(cell):   continue                                                                                                                                                        
          value = 10**cell * 100
          if value < 0.1:   continue                         
          color = "black" if value < 10 else "white"
          if value >= 1:  str_val = str(sigfig.round(value, 2))
          else:  str_val = str(round(value, 1))   
          if str_val.endswith(".0"):  str_val = str_val[:-2]                                                                                                                                          
          clusterMap.ax_heatmap.text(j+0.5, i+0.5, str_val, ha='center', va='center', color=color, fontsize=20)


    clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
    # clusterMap.figure.subplots_adjust(left=0.18, right=0.82, bottom=0.15, top=0.95)
    # if "Methanogens" not in title:  clusterMap.figure.suptitle(title, fontsize=30)
    # else:   clusterMap.figure.suptitle(title, x=0.9, ha='right', fontsize=30)
    clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=24, rotation=0)
    clusterMap.ax_heatmap.set_xticklabels(clusterMap.ax_heatmap.get_xticklabels(), fontsize=24, rotation=70, ha='right', rotation_mode='anchor')
    clusterMap.ax_heatmap.set_xlabel("Days after inoculation", fontsize=32, labelpad=20)
    # clusterMap.ax_row_dendrogram.set_ylabel("Taxonomy", fontsize=16)
    # clusterMap.ax_row_dendrogram.set_title("Taxonomy", fontsize=16, loc='center')
    clusterMap.ax_row_dendrogram.xaxis.set_visible(False)
    # clusterMap.ax_row_dendrogram.set_xlabel("Taxonomy", fontsize=16)
    clusterMap.ax_row_dendrogram.text(
        0.65, -0.02,  # x, y in axes coordinates
        "Taxonomical tree",
        fontsize=24,
        ha='center',
        transform=clusterMap.ax_row_dendrogram.transAxes
    )

    # --- swap y-axes: labels to the left, dendrogram to the right                                                                                                              
    clusterMap.ax_heatmap.yaxis.tick_left()                                                                                                                                     
    clusterMap.ax_heatmap.yaxis.set_label_position("left")
                                                                                                                                                                                
    # Desired x-layout across the figure (0..1):                                                                                                                                
    #   [ 0.00 .. 0.22 ]  left margin for y-tick labels                                                                                                                         
    #   [ 0.22 .. 0.75 ]  heatmap                                                                                                                                               
    #   [ 0.75 .. 0.90 ]  row dendrogram                            
    #   [ 0.90 .. 1.00 ]  right margin (colorbar, title, slack)                                                                                                                 
    label_right   = 0.22                                                                                                                                                        
    heatmap_right = 0.68     # was 0.70 — pull in to make room for strip
    dendro_left   = 0.72     # was 0.71 — bumped right so strip fits in 0.68 → 0.72                                                                                             
    dendro_right  = 0.85                                                                                                                                                     
                                                                                                                                                                                
    hm_pos   = clusterMap.ax_heatmap.get_position()                                                                                                                             
    dend_pos = clusterMap.ax_row_dendrogram.get_position()

    _cbar_w = 0.025
    _cbar_h = hm_pos.height * 0.6
    clusterMap.ax_cbar.set_position([
        dendro_right + 0.05,
        hm_pos.y0 + (hm_pos.height - _cbar_h) / 2,
        _cbar_w,
        _cbar_h,
    ])                                                                                                                                 
    clusterMap.ax_heatmap.set_position([                                                                                                                                        
        label_right, hm_pos.y0,
        heatmap_right - label_right, hm_pos.height,                                                                                                                             
    ])                                                              
    clusterMap.ax_row_dendrogram.set_position([
        dendro_left, dend_pos.y0,
        dendro_right - dendro_left, dend_pos.height,                                                                                                                            
    ])
    clusterMap.ax_row_dendrogram.invert_xaxis()                                                                                                                                 
                                                                    
    # Move colorbar off the left edge — it was colliding with the relocated y-labels.                                                                                           
    # Park it in the top-right, vertical orientation.
    # Align the col-dendrogram (Shannon plot lives here) to the new heatmap x-extent                                                                                            
    gap = 0.03   # figure-fraction breather between heatmap and Shannon plot                                                                                                    
    col_dend_pos = clusterMap.ax_col_dendrogram.get_position()                                                                                                                  
    clusterMap.ax_col_dendrogram.set_position([                                                                                                                                 
        label_right,                                                
        col_dend_pos.y0 + gap,                       # shifted up                                                                                                               
        heatmap_right - label_right,                                                                                                                                            
        col_dend_pos.height,                          # original height — no compression
    ])          


    # Index by BOTH the full key AND the stripped form so it works regardless of                                                                                                
    # whether the heatmap was built ASV-level (".N" suffix) or genus-level (split).
    organisms_to_highlight = ["Methanobacterium", "Methanosarcina", "Methanobacteriaceae"]                                                                                      
    iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", 'r'))                                                                                                
    ID_levels = {}                                                                                                                                                              
    for k, v in iterativeID_levels.items():                         
        ID_levels[k] = v                                                                                                                                                        
        ID_levels.setdefault(k.split(".")[0], v)                                                                                                                                
    
    for label in clusterMap.ax_heatmap.get_yticklabels():                                                                                                                       
        text = label.get_text()                                     
        if any(x in text for x in organisms_to_highlight):                                                                                                                      
            # label.set_fontsize(26 * 1.2)                            
            label.set_fontweight('bold')                                                                                                                                        
        if ID_levels.get(text) == "Genus":
            label.set_fontstyle("italic")       

    # Color strips sandwiched between tick labels and the heatmap                                                                                                               
    hm_pos = clusterMap.ax_heatmap.get_position()                                                                                                                               
    fig_w  = clusterMap.figure.get_figwidth()    # inches
    fig_h  = clusterMap.figure.get_figheight()                                                                                                                                  
    strip_w = 0.015   # row_colors width in figure fraction                                                                                                                      
    strip_h = 0.015   # col_colors height in figure fraction         
    # Row strip: between heatmap right edge and row dendrogram left edge
    clusterMap.ax_row_colors.set_position([                                                                                                                                     
        hm_pos.x1 + 0.005,        # tiny breather from heatmap
        hm_pos.y0,                                                                                                                                                              
        strip_w,                                                    
        hm_pos.height,                                                                                                                                                          
    ])                     
    y_pad_pts = strip_w * fig_w * 72 + 12   # strip width in points + breather
    clusterMap.ax_heatmap.tick_params(axis='y', pad=y_pad_pts)                                         
    # Col strip: move from the top of the heatmap to the bottom (between heatmap and x-labels)                                                                                  
    # clusterMap.ax_col_colors.set_position([
    #     hm_pos.x0,                                                                                                                                                              
    #     hm_pos.y0 - strip_h,                                        
    #     hm_pos.width,                                                                                                                                                           
    #     strip_h,
    # ])     
    #                           
    # Build phylum → color from the rows we just rendered   
    # Make row_colors index-keyable regardless of how it came in                                                                                                                
    if isinstance(row_colors, Series):                                                                                                                                          
        rc = row_colors
    else:                                   # list aligned with df.index                                                                                                        
        rc = Series(list(row_colors), index=df.index)                                                                                                                       
    phylum_color = {}
    for idx in df.index:                                                                                                                                                   
        parts = str(taxonomies.get(idx, "")).split("|")             
        if len(parts) < 2:  continue
        phylum = parts[1]                                                                                                                                                       
        if phylum in ("None", "", "Unknown"): continue                                                                                                                                                            
        color = rc.get(idx)
        if color is None:  continue                                                
        phylum_color.setdefault(phylum, color)      
    print("phylum_color", phylum_color)   # empty {}                                                                                                                          
                                                                    
    # Sort archaea first, then bacteria (matches your network-plot ordering)                                                                                                    
    archaea_markers = ("archaeo", "euryarchaeota", "crenarchaeota", "thaumarchaeota",
                        "halobacterota", "methanobacteriota", "micrarchaeota", "nanoarchaeota")                                                                                  
    def is_archaea(p): return any(m in p.lower() for m in archaea_markers)                                                                                                      
                                                                                                                                                                                
    archaea  = sorted(p for p in phylum_color if is_archaea(p))                                                                                                                 
    bacteria = sorted(p for p in phylum_color if not is_archaea(p))  
    # print(archaea, bacteria)                                                                                                           
                                                                                                                                                                                
    handles = []                                                    
    if archaea:                                                                                                                                                                 
        handles.append(Patch(color="none", label=r"$\bf{Archaea}$"))
        handles += [Patch(facecolor=phylum_color[p], label=p) for p in archaea]                                                                                                 
    if bacteria:                                                                                                                                                                
        handles.append(Patch(color="none", label=r"$\bf{Bacteria}$"))                                                                                                           
        handles += [Patch(facecolor=phylum_color[p], label=p) for p in bacteria]                                                                                                
                                                                    
    clusterMap.figure.legend(
        handles=handles,                                                                                                                                                        
        title="Phylum",                                                                                                                                                         
        title_fontsize=20,
        fontsize=16,                                                                                                                                                            
        loc="upper right",                 # anchor the legend's upper-RIGHT corner
        bbox_to_anchor=(0.07, 1.0),       # ...just to the left of the colorbar's x=0.09                                                                                       
        frameon=True,                                                                                                                                                           
        borderaxespad=0.5,                                                                                                                                                      
        handlelength=1.5,                                                                                                                                                       
        handletextpad=0.6,                                          
    )                 

    for spine in clusterMap.ax_heatmap.spines.values():                                                                                                                         
        spine.set_visible(True)                                     
        spine.set_edgecolor("black")                                                                                                                                        
        spine.set_linewidth(1)                                                                                                         
                                                                
    # Colorbar: dedicated slot in the far-right margin, with proper height                                                                                                      
    # clusterMap.ax_cbar.set_position([0.93, 0.25, 0.025, 0.55])

    # clusterMap.figure.tight_layout()
    clusterMap.figure.savefig(f"abundance_heatmaps/{title.lower().replace(' ', '_')}.png", bbox_inches="tight", dpi=800)
    clusterMap.figure.savefig(f"abundance_heatmaps/{title.lower().replace(' ', '_')}.svg", bbox_inches="tight")



from numpy import log
from json import load, dump

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))

def shannon_index(abundances):
    return -sum([abundance * log(abundance) for abundance in abundances if abundance > 0])

abundances_js = load(open("model_inputs/abundances.json", 'r'))
shannon_indices = {sample_days[s]: shannon_index(list(abundances.values())) for s, abundances in abundances_js.items() if s in sample_days}
display(shannon_indices)

# Create the Methanogens heatmap
dic = {}
taxonomies = {}
level = "Genus"
for seq, row in total_df.iterrows():
    day = sample_days.get(row["sample"])
    if row["Kingdom"] == "Bacteria" or day is None:  continue
    
    abundance = row["rel_ab"]
    if abundance == 0:  continue
    iterativeID = iterativeIDs.get(seq) or "Archaea.99"
    if isinstance(iterativeID, str):
        iterativeID = iterativeID.split(".")[0]
    taxonomies.setdefault(iterativeID, "|".join([str(row[l]) for l in taxonomic_levels
                                    if taxonomic_levels.index(l) <= taxonomic_levels.index(level)]))
    dic.setdefault(day, {})
    dic[day].setdefault(iterativeID, 0)
    dic[day][iterativeID] += abundance

new_dic = {}
for day, content in dic.items():
    new_dic[day] = {genus: log10(abund) for genus, abund in content.items()}

def change_columns(df):
    return df[[c for c in sample_days.values() if c in df.columns]]
    # return df[[c for c in sample_days if c in df.columns]]
    
df = change_columns(DataFrame(new_dic))
display(df)
df = df.astype(float).replace([inf, -inf], nan)#.fillna(0)
taxonomy_series = Series({idx: taxonomies.get(idx, f"Unknown|{idx}") for idx in df.index})

arr = df.values.astype(float)
row_means = nanmean(arr, axis=1, keepdims=True)
# Drop rows whose label is blank / nan / "Unknown"
def _row_keep(_idx, _ts):
    s = str(_idx).strip()
    if not s or s.lower() in ("nan", "none", "") or s.startswith(("Unknown", "None", "nan.")):
        return False
    parts = str(_ts.get(_idx, "")).split("|")
    if len(parts) < 2:
        return False
    phylum = parts[1].strip()
    return bool(phylum) and phylum.lower() not in ("nan", "none", "unknown", "")
df = df.loc[[_idx for _idx in df.index if _row_keep(_idx, taxonomy_series)]]
taxonomy_series = taxonomy_series.loc[df.index]
arr = df.values.astype(float)
row_means = nanmean(arr, axis=1, keepdims=True)
arr_filled = where(isnan(arr), row_means, arr)
row_linkage = hierarchy.linkage(arr_filled, method='average', metric='euclidean')
create_heatmap(df, taxonomy_series, "Methanogens Abundances (% abundance)", shannon_indices, row_linkage)


# Create a single Genus-level heatmap
#TODO:  Make this figure logarithmic to ensure that all organisms can be perceived
dic, taxonomies = {}, {}
for seq, row in total_df.iterrows():
    day = sample_days.get(row["sample"])
    if day is None:  continue
    iterativeID = iterativeIDs.get(seq)
    taxonomies.setdefault(iterativeID, "|".join([str(row[l]) for l in taxonomic_levels
                            if taxonomic_levels.index(l) <= taxonomic_levels.index("Genus")]))
    dic.setdefault(day, {})
    dic[day].setdefault(iterativeID, 0)
    dic[day][iterativeID] += row["rel_ab"]

nonzero_per_day = {
    day: dict(sorted({k:v for k,v in org_dict.items() if v>0}.items(), key=lambda item: item[1], reverse=True))
    for day, org_dict in dic.items()
}
dump(nonzero_per_day, open("model_inputs/nonzero_per_day.json", "w"), indent=2)
top_per_day = {}
all_orgs = {}
topNum = 10
top10_all_days = []
for day, org_dict in nonzero_per_day.items():
    top10_all_days.extend(list(org_dict.keys())[:topNum])
top10_all_days = list(set(top10_all_days))
for day, org_dict in nonzero_per_day.items():
    # orgs = dict(list(org_dict.items())[:topNum])
    top_per_day.setdefault(day, {})
    top_per_day[day] = {org:log10(v) for org,v in org_dict.items() if org in top10_all_days}
    # for org, abundance in org_dict.items():
    #     if org not in top10_all_days:  continue
    #     top_per_day[day][org] = log10(abundance)
    # orgs = dict(list(org_dict.items())[:topNum])
    # top_per_day[day] = {k:log10(v) for k,v in orgs.items()}
    # all_orgs.update(orgs)
taxonomies = {org:taxa for org, taxa in taxonomies.items() if org in top10_all_days}
display(top_per_day)
display(taxonomies)

df = change_columns(DataFrame(top_per_day))
df = df.astype(float).replace([inf, -inf], nan)#.fillna(0)
taxonomy_series = Series({idx: taxonomies.get(idx, f"Unknown|{idx}") for idx in df.index})
display(df)

arr = df.values.astype(float)
row_means = nanmean(arr, axis=1, keepdims=True)
# Drop rows whose label is blank / nan / "Unknown"
def _row_keep(_idx, _ts):
    s = str(_idx).strip()
    if not s or s.lower() in ("nan", "none", "") or s.startswith(("Unknown", "None", "nan.")):
        return False
    parts = str(_ts.get(_idx, "")).split("|")
    if len(parts) < 2:
        return False
    phylum = parts[1].strip()
    return bool(phylum) and phylum.lower() not in ("nan", "none", "unknown", "")
df = df.loc[[_idx for _idx in df.index if _row_keep(_idx, taxonomy_series)]]
taxonomy_series = taxonomy_series.loc[df.index]
arr = df.values.astype(float)
row_means = nanmean(arr, axis=1, keepdims=True)
arr_filled = where(isnan(arr), row_means, arr)
row_linkage = hierarchy.linkage(arr_filled, method='average', metric='euclidean')
create_heatmap(df, taxonomy_series, f"Top {topNum} ASVs (% abundance)", shannon_indices, row_linkage)

### table of iterative IDs

In [ ]:
all_top10 = {
    'Methanobacterium.1': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'Methanobacterium.2': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'Methanobacteriaceae.1': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|nan',
    'Mesotoga.1': 'Bacteria|Thermotogota|Thermotogae|Kosmotogales|Kosmotogaceae|Mesotoga',
    'Lentimicrobium.1': 'Bacteria|Bacteroidota|Bacteroidia|Sphingobacteriales|Lentimicrobiaceae|Lentimicrobium',
    'Aminivibrio.1': 'Bacteria|Synergistota|Synergistia|Synergistales|Synergistaceae|Aminivibrio',
    'midas_g_94288.1': 'Bacteria|Firmicutes|Clostridia|Gracilibacteraceae|Lutispora|midas_g_94288',
    'Desulfovibrio.1': 'Bacteria|Desulfobacterota|Desulfovibrionia|Desulfovibrionales|Desulfovibrionaceae|Desulfovibrio',
    'Methanobacterium.3': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'Petrimonas.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Dysgonomonadaceae|Petrimonas',
    'Burkholderiales.1': 'Bacteria|Proteobacteria|Gammaproteobacteria|Burkholderiales|nan|nan',
    'midas_g_9269.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Paludibacteraceae|midas_g_9269',
    'Methanobacterium.5': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'Proteiniphilum.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Dysgonomonadaceae|Proteiniphilum',
    'midas_g_36215.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Rikenellaceae|midas_g_36215',
    'Methanobacterium.6': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'Enterobacteriaceae.1': 'Bacteria|Proteobacteria|Gammaproteobacteria|Enterobacterales|Enterobacteriaceae|nan',
    'EBM-39.1': 'Bacteria|Synergistota|Synergistia|Synergistales|Synergistaceae|EBM-39',
    'Dysgonomonas.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Dysgonomonadaceae|Dysgonomonas',
    'Methanobacterium.78': 'Archaea|Euryarchaeota|Methanobacteria|Methanobacteriales|Methanobacteriaceae|Methanobacterium',
    'midas_g_25828.1': 'Bacteria|Bacteroidota|Bacteroidia|Bacteroidales|Prolixibacteraceae|midas_g_25828',
    'Azoarcus.4': 'Bacteria|Proteobacteria|Gammaproteobacteria|Burkholderiales|Rhodocyclaceae|Azoarcus',
    'Desulfovibrio.3': 'Bacteria|Desulfobacterota|Desulfovibrionia|Desulfovibrionales|Desulfovibrionaceae|Desulfovibrio',
    'Desulfomicrobium.1': 'Bacteria|Desulfobacterota|Desulfovibrionia|Desulfovibrionales|Desulfomicrobiaceae|Desulfomicrobium',
    'Stappia.1': 'Bacteria|Proteobacteria|Alphaproteobacteria|Rhizobiales|Stappiaceae|Stappia',
    'midas_g_1799.7': 'Bacteria|Firmicutes|Clostridia|Proteinivoracales|midas_f_1799|midas_g_1799',
    'midas_g_7.1': 'Bacteria|Firmicutes|Dethiobacteria|Dethiobacterales|Dethiobacteraceae|midas_g_7',
    'Dethiobacteraceae.4': 'Bacteria|Firmicutes|Dethiobacteria|Dethiobacterales|Dethiobacteraceae|nan',
    'Soehngenia.2': 'Bacteria|Firmicutes|Clostridia|Peptostreptococcales-Tissierellales|Family_XI|Soehngenia'
    }

from pandas import DataFrame
from json import load

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
IDasvs = {}
for k, v in iterativeIDs.items():
    IDasvs.setdefault(v, []).append(k)

# develop the dictionary for Heather's table
total_dic = {"ASV": [], "Iterative ID": [], "MiDAS Taxonomy": []}
for iterativeID, taxonomy in all_top10.items():
    if iterativeID not in IDasvs:  # stale label absent from current map
        continue
    total_dic["Iterative ID"].append(iterativeID)
    total_dic["MiDAS Taxonomy"].append(taxonomy.replace("|", " "))
    total_dic["ASV"].append(IDasvs.get(iterativeID)[0])
# total_dic

df = DataFrame(total_dic).set_index("ASV")
df.to_csv("model_inputs/asv_IDs.csv")
df

In [ ]:
# from pandas import read_csv, DataFrame, Series, concat
# import seaborn as sns
# from matplotlib import pyplot
# from json import load
# from numpy import inf, nan, empty, logspace
# from scipy.spatial.distance import squareform
# from scipy.cluster.hierarchy import linkage
# from collections import Counter

# # taxonomy = load(open("/Users/andrewfreiburger/Documents/Research/MicrobiomeNotebooks/digestor/new_abundances.json", 'r'))

# def taxonomy_linkage(taxonomy_series, ranks=('kingdom','phylum','class','order','family','genus','species'),
#                      method: str = 'average'):
#     """
#     Build a SciPy linkage matrix from a taxonomy Series by defining
#     a simple distance: (#ranks) - (#equal ranks at the same positions).

#     Parameters
#     ----------
#     taxonomy_series : pd.Series
#         Index order must match df.index (rows). Values are taxonomy strings
#         like "k__Bacteria;p__Firmicutes;c__Bacilli;...".
#     ranks : tuple or int
#         Number of ranks considered. If tuple, its length is used.
#     method : str
#         Linkage method for scipy.cluster.hierarchy.linkage.

#     Returns
#     -------
#     Z : np.ndarray
#         Linkage matrix suitable for seaborn.clustermap(row_linkage=Z).
#     """
#     # Normalize rank count
#     ranks_n = len(ranks) if not isinstance(ranks, int) else int(ranks)

#     # Split and standardize to fixed-length lists (pad/truncate)
#     parts = (
#         taxonomy_series.fillna("")
#         .astype(str)
#         .str.split(r"[;|,]", regex=True)
#         .apply(lambda lst: [x.strip() for x in (lst or [])])
#         .apply(lambda lst: (lst + [""] * ranks_n)[:ranks_n])
#     )

#     n = len(parts)
#     if n <= 1:
#         # Nothing to cluster; return empty linkage
#         return empty((0, 4), dtype=float)

#     # Build condensed distance vector: distance = ranks_n - (#equal positions)
#     D_condensed = []
#     for i in range(n - 1):
#         ai = parts.iloc[i]  # POSitional access fixes KeyError
#         for j in range(i + 1, n):
#             aj = parts.iloc[j]
#             same = sum(a == b and a != "" and b != "" for a, b in zip(ai, aj))
#             D_condensed.append(ranks_n - same)

#     # Convert to linkage (hierarchical clustering tree)
#     Z = linkage(D_condensed, method=method)
#     return Z


# def concatenate_taxonomy(df, level):
#     dic = {}
#     for index, row in df.iterrows():
#         dic["|".join([str(row[l]) for l in taxonomic_levels
#                     if taxonomic_levels.index(l) <= taxonomic_levels.index(level)])] = index
#     return Series(dic)

# def change_columns(df):
#     return df[[c for c in sample_days.values() if c in df.columns]]
#     # return df[[c for c in sample_days if c in df.columns]]


# from matplotlib.colors import LinearSegmentedColormap

# # Define a single-color gradient colormap (white → blue)
# new_cmap = LinearSegmentedColormap.from_list("NewMap", [(0, "skyblue"), (1, "red")])


# from pandas import read_excel
# from json import dump
# # ab = read_csv("GAME Sequencing/ASV_abundance_wideform.csv").set_index("seq")
# taxonomic_levels = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
# # sample_order = ["10AB", "A34", "B34", "C3", "D34", "E34", "F34", "G12", "G3", "H34", "I34",
# #                 "J12", #"J34",
# #                 "K12", "L12", "M12", "N12", "P12"]
# sample_days = {
#     "5-10": 0,
#     "H": 140,
#     "Ha": 147,
#     "I": 149,
#     "Ia": 153,
#     "J": 161,
#     "Ja": 168,
#     "Jb": 174,
#     "K": 182,
#     "L": 189,
#     "M": 210,
#     "N": 238,
#     "Na": 262,
#     "P": 283,
#     "Q": 300,
#     "Qa": 315,
# }

# og = read_csv("model_inputs/").set_index("seq")
# sample_dfs = []
# for sample in sample_days:
#     # print(sample)
#     sample_og = og[og["sample"] == sample]
#     # display(sample_og)
#     sample_og["rel_ab"] = sample_og["rel_ab"] / sample_og["rel_ab"].sum()
#     sample_dfs.append(sample_og)
# og = concat(sample_dfs)
# # og["rel_ab"] /= og["rel_ab"].sum()
# # max_abundance = og["rel_ab"].max()
# # print(max_abundance)
# # display(og)
# intervals = [0] + list(logspace(-2, -.3, 5))
# print(intervals)
# for i, abundance_limit in enumerate(intervals):
#     if abundance_limit == 0:  continue
#     previous_limit = intervals[i-1]
#     print(previous_limit, abundance_limit)
#     abundance_range = f"{round(previous_limit, 2)}-{round(abundance_limit, 2)}"
#     for level in taxonomic_levels:
#         ab = og.copy()
#         # ab = ab.set_index(level).drop([l for l in taxonomic_levels if l != level], axis=1)#.groupby(level).sum()
#         # display(ab)

#         dic = {}
#         taxonomies = {}
#         for seq, row in ab.iterrows():
#             s = row["sample"]
#             if s == "J34":  continue
            
#             day = sample_days[s]
#             abundance = row["rel_ab"]
#             # print(abundance)
#             if abundance <= previous_limit or abundance > abundance_limit:  continue
#             # asvs_used.append(row["seq"])
#             taxonomies[row[level]] = "|".join([str(row[l]) for l in taxonomic_levels
#                                 if taxonomic_levels.index(l) <= taxonomic_levels.index(level)])
#             # taxonomies.append("|".join([str(row[l]) for l in taxonomic_levels
#             #                     if taxonomic_levels.index(l) <= taxonomic_levels.index(level)]))
#             dic.setdefault(day, {})
#             dic[day].setdefault(row[level], 0)
#             dic[day][row[level]] += abundance

#         # display(dic)
#         if dic == {}:
#             print(f"The {level} does not have organisms in {abundance_range}")
#             continue
#         elif len(taxonomies) == 1:
#             print(f"The {level} only has one organism in {abundance_range}:  {dic} and {taxonomies}")
#             continue

#         # display(dic)

#         df = change_columns(DataFrame(dic))
#         df = df.astype(float).replace([inf, -inf], nan).fillna(0)
#         # display(df)
#         # display(df.to_numpy())
#         max_abundance = max(list(df.to_numpy().flatten()))
#         # display(df)
#         # clustermap = ClusterHeatmap(df, level, ) # Series(dict(Counter(taxonomies))))
#         clusterMap = sns.clustermap(df, 
#                                     # row_colors=row_colors,
#                                     cmap=new_cmap,
#                                     vmin=0,
#                                     vmax=max_abundance,
#                                     col_cluster=False, figsize=(20, 20),
#                                     row_linkage=taxonomy_linkage(Series(taxonomies)),
#                                     dendrogram_ratio=(.1, .2))
#         clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
#         clusterMap.figure.suptitle(f"{level.capitalize()} {abundance_range} Abundances", fontsize=30)
#         clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=20, rotation=0)
#         clusterMap.ax_heatmap.set_xticklabels(clusterMap.ax_heatmap.get_xticklabels(), fontsize=24)
#         clusterMap.ax_heatmap.set_xlabel("Days after inoculation", fontsize=30)
#         clusterMap.figure.tight_layout()
#         clusterMap.figure.savefig(f"abundance_heatmaps/{level}_{abundance_range}_abundances.png")
#     # break

# Correlate member abundances with methane

In [ ]:
merged_correlations = {}
for index, row in merged_normalized_abundances.iterrows():
    corr = row.corr(normalized_methane)
    merged_correlations[index] = corr
    # print(f"{index} correlates {corr} with methane")

merged_correlations = dict(sorted(merged_correlations.items(), key=lambda item: item[1], reverse=True))
merged_correlations

In [ ]:
from pandas import DataFrame, Series, isna
from json import load, dump

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
abundances = load(open("model_inputs/abundances.json", 'r'))
asvSet_abundances = load(open(f"modeling_files/ASVset_abundances.json", 'r'))

omitted_columns = {"all_relevant_samples": ["5-10"],   # exclude day-0 inoculum from correlation analysis
                  }

from scipy.stats import spearmanr
def correlations(ser1, ser2):
    aligned1, aligned2 = ser1.align(ser2, join='inner')
    if len(set(aligned1)) == 1 or len(set(aligned2)) == 1:
        global constant_vals
        constant_vals += 1
        return float('nan'), float('nan')
    return spearmanr(aligned1, aligned2)

ommitted = omitted_columns["all_relevant_samples"]   # used by the upfront drop below; overwritten in the for-loop
abundances_df = DataFrame(abundances).drop(ommitted, axis=1)
abundances_df = abundances_df.loc[:, (abundances_df.fillna(0) > 0).sum() >= 5]
ASVset_abundances_df = DataFrame(asvSet_abundances).drop(ommitted, axis=1).fillna(0)
ASVset_abundances_df = ASVset_abundances_df.loc[:, ASVset_abundances_df.notna().sum() >= 5]
print(abundances_df.shape)
summary_samples = load(open("model_inputs/measurements/summary_samples.json", 'r'))
# print(list(summary_samples.keys()))
print([summary_samples[s]["CH4 Production (mol/min)"] for s in summary_samples.keys()])
print([summary_samples[s]["H2 delivery rate (mol/min)"] for s in summary_samples.keys()])
for dataName, term in {"CH4%": "Gas Composition (% CH4)", "CH4_mol": "CH4 Production (mol/min)",
                        "CO2_BT": "CO2 breakthrough (mol/min)", "CO2_feed": "CO2 delivery rate (mol/min)",
                        "H2_BT": "H2 breakthrough (mol/min)", "H2_feed": "H2 delivery rate (mol/min)"}.items():
    for name, ommitted in omitted_columns.items():
        constant_vals = 0
        sample_values = {sample: content[term] for sample, content in summary_samples.items()
                         if sample not in ommitted}
        print(dataName)
        
        # ASV correlations
        ASV_correlations = {}
        constant_vals = 0
        for ASV, sampleAbun in abundances_df.iterrows():
            ID = iterativeIDs.get(ASV, ASV)
            correlation, p = correlations(sampleAbun, Series(sample_values))
            if isna(correlation):  continue
            ASV_correlations[ID] = {"correlation": correlation, "p_value": p}
        
        ASV_correlations = dict(sorted(ASV_correlations.items(), key=lambda item: item[1]["correlation"], reverse=True))
        print(f"Constant values: {constant_vals}")
        dump(ASV_correlations, open(f"modeling_files/correlations/ASV_correlations_{name}_{dataName}.json", 'w'))
        
        # IterativeID correlations
        IterativeID_correlations = {}
        for ASV, sampleAbun in ASVset_abundances_df.iterrows():
            ID = iterativeIDs.get(ASV, ASV)
            correlation, p = correlations(sampleAbun, Series(sample_values))
            if isna(correlation):  continue
            IterativeID_correlations[ID] = {"correlation": correlation, "p_value": p}
        
        IterativeID_correlations = dict(sorted(IterativeID_correlations.items(), key=lambda item: item[1]["correlation"], reverse=True))
        dump(IterativeID_correlations, open(f"modeling_files/correlations/IterativeID_correlations_{name}_{dataName}.json", 'w'))
        print(IterativeID_correlations)
        
        # ASVset correlations
        ASVset_abundances_df.index = [iterativeIDs.get(ASV,ASV).split(".")[0] for ASV in ASVset_abundances_df.index]
        ASVset_abundances_df.groupby(ASVset_abundances_df.index).sum()
        ASVset_correlations = {}
        for genus, sampleAbun in ASVset_abundances_df.iterrows():
            correlation, p = correlations(sampleAbun, Series(sample_values))
            if isna(correlation):  continue
            ASVset_correlations[genus] = {"correlation": correlation, "p_value": p}
        
        ASVset_correlations = dict(sorted(ASVset_correlations.items(), key=lambda item: item[1]["correlation"], reverse=True))
        dump(ASVset_correlations, open(f"modeling_files/correlations/ASVset_correlations_{name}_{dataName}.json", 'w'))

In [ ]:
from matplotlib import colors, pyplot, patches
from matplotlib.patches import Patch
from pandas import DataFrame, Series, set_option, read_csv
from collections import Counter
from numpy import inf, nan
import seaborn as sns
from json import load
from glob import glob
from statsmodels.stats.multitest import multipletests

set_option('display.max_rows', None)

# ab = read_csv("GAME Sequencing/ASV_abundance_wideform.csv").set_index("seq")
taxonomic_levels = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
sample_days = {
    "5-10": 0,
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}

# order = ["Days 150-300 $H_2$ loading", "Days 150-300 $H_2$ BT", "Days 189-238 $H_2$ loading", "Days 189-238 $H_2$ BT"]
order = ["all samples $H_2$ feed", "all samples $H_2$ BT",
        "all samples $CO_2$ feed", "all samples $CO_2$ BT",
        "all samples $CH_4$%", "all samples $CH_4$ mol"]
correlations, pvals = {}, {}
for k in order:
    correlations.setdefault(k, {})
    pvals.setdefault(k, {})
# for cor in glob("modeling_files/correlations/IterativeID_correlations_Days*.json"):
# for cor in glob("modeling_files/correlations/IterativeID_correlations_all_relevant_samples_H2*.json"):
for cor in glob("modeling_files/correlations/ASV_correlations_all_relevant_samples_*.json"):
    print(cor)
    name = cor.split("/")[-1].split(".")[0].split("_correlations_")[1].replace("_", ' ')
    name = name.replace("relevant ", "").replace("phase ", "")
    name = name.replace("H2","$H_2$").replace("CO2","$CO_2$").replace("CH4","$CH_4$")
    print(name)
    # if "238" in name:  continue
    content = load(open(cor, 'r'))
    # print(name, content)
    correlations[name].update({k: v["correlation"] for k, v in content.items()})
    pvals[name].update({k: v["p_value"] for k, v in content.items()})

# display(correlations)
# break
q_vals = {}
for name, content in pvals.items():
    pvals_list = list(content.values())
    q_vals[name] = multipletests(pvals_list, alpha=0.05, method='fdr_bh')
    print(sum(q_vals[name][0]))
new_correlations = {name: dict(inner) for name, inner in correlations.items()}                                                                                              
new_pvals        = {name: dict(inner) for name, inner in pvals.items()}
for name, content in pvals.items():
    for i, (k, v) in enumerate(content.items()):
        if not q_vals[name][0][i]:
            new_correlations[name].pop(k)
            new_pvals[name].pop(k)

display(new_correlations)
display(new_pvals)
# # keys = [(name, k) for name, inner in pvals.items() for k in inner]
# # flat_pvals = [pvals[name][k] for name, k in keys]
# # print(len(set(flat_pvals)))
# # reject, qvals, _, _ = 
# # Build filtered dicts containing only the surviving entries
# filtered_correlations = {name: {} for name in correlations}
# filtered_pvals = {name: {} for name in pvals}
# filtered_qvals = {name: {} for name in pvals}
# for (name, k), keep, q in zip(keys, reject, qvals):
#     if keep:
#         filtered_correlations[name][k] = correlations[name][k]
#         filtered_pvals[name][k] = pvals[name][k]
#         filtered_qvals[name][k] = q
# filtered_correlations = {n: d for n, d in filtered_correlations.items() if d}
# filtered_pvals       = {n: d for n, d in filtered_pvals.items()       if d}
# filtered_qvals       = {n: d for n, d in filtered_qvals.items()       if d}

df = DataFrame(new_correlations).fillna(0)
# df = df[["stable CH4%", "stable CH4 mol", "stable H2 BT", "all samples CH4%", "all samples CH4 mol", "all samples H2 BT"]]
pval_matrix = DataFrame(new_pvals)#.fillna(1)
# df.drop(["Days 189-238 $H_2$ loading", "Days 189-238 $H_2$ BT"], axis=1, inplace=True)
_rename_full = {
    "all samples $H_2$ feed":   r"$H_2$ loading $\left(\frac{mol}{min}\right)$",
    "all samples $H_2$ BT":     r"$H_2$ breakthrough $\left(\frac{mol}{min}\right)$",
    "all samples $CO_2$ feed":  r"$CO_2$ loading $\left(\frac{mol}{min}\right)$",
    "all samples $CO_2$ BT":    r"$CO_2$ breakthrough $\left(\frac{mol}{min}\right)$",
    "all samples $CH_4$%":      r"$CH_4$ composition $\left(\%\right)$",
    "all samples $CH_4$ mol":   r"$CH_4$ production $\left(\frac{mol}{min}\right)$",
}
_rename_short = {k: v.split(" $\\left")[0] for k, v in _rename_full.items()}
df.rename(columns=_rename_full, inplace=True)
pval_matrix.rename(columns=_rename_short, inplace=True)
reduced = True
if reduced:
    df = df[(df != float(0)).any(axis=1)] # filtering empty, no correlated, rows
# display(df)

total_df = read_csv("model_inputs/total.csv").set_index("seq")
orgs = {}
for i in df.index:
    orgs.setdefault(i.split(".")[0], []).append(i)
taxonomies = {}
level = "Genus"
for seq, row in total_df.iterrows():
    day = sample_days.get(row["sample"])
    taxonomy = []
    for l in reversed(taxonomic_levels):
        taxa = str(row[l])
        # taxonomy.append(taxa)
        IDs = orgs.get(taxa)
        if IDs is not None: break

    if day is None or row["rel_ab"] == 0 or IDs is None:  continue
    taxonomy_string = "|".join([str(row[l]) for l in taxonomic_levels
                                if taxonomic_levels.index(l) <= taxonomic_levels.index(level)])
    for ID in IDs:
        taxonomies.setdefault(ID, taxonomy_string)
taxonomy_series = Series({ID: taxonomies.get(ID, f"Unknown|{ID.split('.')[0]}") for ID in df.index}) #Series(taxonomies) # 
# print(taxonomy_series)
min_max = (round(df.min().min(),1), round(df.max().max(), 1))
clusterMap = sns.clustermap(df,
                            cmap="coolwarm_r",
                            norm=colors.TwoSlopeNorm(vmin=min_max[0], vcenter=0, vmax=min_max[1]),
                            col_cluster=False,
                            clip_on=True,
                            # row_linkage=taxonomy_linkage(taxonomy_series),
                            figsize=(20, 80), cbar_kws={"label": "Correlation"})
labelsize = 70
clusterMap.ax_heatmap.set_xlabel("Operational metric", fontsize=labelsize, labelpad=40)
# clusterMap.ax_heatmap.tick_params(axis='y', labelrotation=90)
clusterMap.ax_heatmap.set_ylabel('ASV', fontsize=labelsize, labelpad=40)
clusterMap.ax_col_dendrogram.set_visible(False)
clusterMap.ax_row_dendrogram.set_visible(False)
clusterMap.ax_heatmap.yaxis.tick_left()                                                                                                                                     
clusterMap.ax_heatmap.yaxis.set_label_position("left")


hm = clusterMap.ax_heatmap.get_position()                                                                                                                                   
rd = clusterMap.ax_row_dendrogram.get_position()
clusterMap.ax_heatmap.set_position([rd.x0, hm.y0, hm.x1 - rd.x0, hm.height])

# Add text under the row dendrogram
# clusterMap.ax_row_dendrogram.text(
#     0.6, 0,  # x, y position (in axis coordinates)
#     "Taxonomy Tree",
#     transform=clusterMap.ax_row_dendrogram.transAxes,
#     fontsize=25,
#     ha='center',
#     va='top'
# )

cbar = clusterMap.ax_cbar
ticks = cbar.get_yticks()[::2]
ticks[-1] = min_max[1]
ticks[0] = min_max[0]
ticks = [round(t, 1) for t in ticks]
cbar.set_yticklabels(ticks, fontsize=40)
print(min_max, ticks)
cbar.set_yticks(ticks)  # already there, but ensure
cbar.set_xlabel("")
cbar.set_ylabel(r"Spearman $\rho$", fontsize=60, labelpad=30, rotation=90)
cbar.tick_params(labelsize=40, length=12, width=2)

heatmap_pos = clusterMap.ax_heatmap.get_position()
_cbar_w = 0.03
_cbar_h = heatmap_pos.height * 0.6
clusterMap.ax_cbar.set_position([
    heatmap_pos.x1 + 0.05,
    heatmap_pos.y0 + (heatmap_pos.height - _cbar_h) / 2,
    _cbar_w,
    _cbar_h,
])


## add values text to the elements
# for i in range(clusterMap.data2d.shape[0]):
#     for j in range(clusterMap.data2d.shape[1]):
#         value = clusterMap.data2d.iloc[i, j]
#         if value == 0:  continue
#         color = "black" if abs(value) < 0.7 else "white"
#         clusterMap.ax_heatmap.text(j+0.5, i+0.5, f'{value:.2f}', ha='center',
#                                    va='center', color=color, fontsize=60)

organisms_to_highlight = ["Methanobacterium", "Methanosarcina", "Methanobacteriaceae"]
iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", 'r'))
ID_levels = {k.split(".")[0]: v for k, v in iterativeID_levels.items()}
ylabels = clusterMap.ax_heatmap.get_yticklabels()
for label in ylabels:
    if any([x in label.get_text() for x in organisms_to_highlight]):  # your target labels
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
    text = label.get_text().split(".")[0]
    taxa = ID_levels.get(text)
    if taxa == "Genus":   label.set_fontstyle("italic")
    label.set_rotation(90)
clusterMap.ax_heatmap.set_yticklabels(ylabels)
# pyplot.setp(clusterMap.ax_heatmap.get_yticklabels(), rotation=90)

# highlight specific organisms
dendrogram_row = clusterMap.dendrogram_row.reordered_ind
dendrogram_col = list(range(pval_matrix.shape[1]))  # original column order
pvals_reordered = pval_matrix.iloc[dendrogram_row, dendrogram_col]
ax = clusterMap.ax_heatmap
for i in range(pvals_reordered.shape[0]):
    for j in range(pvals_reordered.shape[1]):
        # if pvals_reordered.iloc[i, j] < 0.05:
        #     ax.add_patch(patches.Rectangle((j, i), 1, 1, fill=False, edgecolor='lightgreen', linewidth=8))
        p = pvals_reordered.iloc[i, j]
        value = clusterMap.data2d.iloc[i, j]
        if value == 0:  continue
        marker = ""
        # if p < 0.01:    marker = "**"
        # elif p < 0.05:  marker = "*"
        color = "black" if abs(value) < 0.7 else "white"
        clusterMap.ax_heatmap.text(
            j + 0.5, i + 0.5, f'{value:.2f}{marker}',
            ha='center', va='center', color=color, fontsize=60
        )

clusterMap.ax_heatmap.set_xticklabels(clusterMap.ax_heatmap.get_xticklabels(), fontsize=50, rotation=80)
clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=50, rotation=0)
# clusterMap.figure.tight_layout()
clusterMap.figure.savefig(f"correlations_heatmap{'' if not reduced else '_reduced'}.png", bbox_inches='tight', dpi=300)
# clusterMap.figure.savefig(f"correlations_heatmap{'' if not reduced else '_reduced'}.svg", bbox_inches='tight')

### All v All correlation matrix

#### Abundance

In [ ]:
from locale import D_FMT
import seaborn as sns
import matplotlib.pyplot as plt
from json import load
from pandas import DataFrame

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
df = DataFrame(load(open("model_inputs/abundances.json", 'r'))).T
columns = []
remove_columns = []
for col in df.columns:
    if df[col].max() < 0.001:
        remove_columns.append(col)
        continue
    columns.append(iterativeIDs[col])
df.drop(remove_columns, axis=1, inplace=True)
df.columns = [ID.split(".")[0] for ID in columns] #columns
df = df.T.groupby(df.columns).sum().T
display(df)
# for sample, row in df.iterrows():
#     df.loc[sample] /= row.sum()
# display(df)
corr_matrix = df.corr("spearman")
display(corr_matrix)

plt.figure(figsize=(150, 120))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.xticks(fontsize=40)
plt.yticks(fontsize=40)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib import colors, pyplot, patches
from matplotlib.patches import Patch
from pandas import Series, DataFrame
from numpy import ones, triu, nan, ones_like
import seaborn as sns
from json import load

days = {
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}

from scipy.stats import spearmanr
def corr_pvalues(df):
    n = df.shape[1]
    pvals = DataFrame(ones((n, n)), index=df.columns, columns=df.columns)
    for i in range(n):
        for j in range(i+1, n):
            _, p = spearmanr(df.iloc[:, i], df.iloc[:, j])
            pvals.iloc[i, j] = p
            pvals.iloc[j, i] = p
    return pvals

significantly_connected_organisms = list(load(open("significantly_connected_organisms.json", 'r')))
significantly_connected_organisms.append("Methanobacteriaceae.1") # hard-coding for Heather
taxa_color_map = load(open(f"iterativeID_color_map.json", "r"))
print(significantly_connected_organisms)

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
df = DataFrame(load(open("model_inputs/abundances.json", 'r'))).T
df.drop(df.index.difference(days.keys()), inplace=True)
df.columns = [iterativeIDs.get(ID, ID) for ID in df.columns] #.split(".")[0]
df.drop([col for col in df.columns if col not in significantly_connected_organisms], axis=1, inplace=True)
df.drop([col for col in df.columns if df[col].max() < 0.0005], axis=1, inplace=True)
total_captured = df.sum(axis=1)
print(sum(total_captured), total_captured)

display(df)
corr_matrix = df.corr("spearman")
display(corr_matrix)
pval_matrix = corr_pvalues(df)
# === BH-FDR on upper-triangle p-values to derive q-value matrix ===
from statsmodels.stats.multitest import multipletests
from numpy import triu_indices as _triu_indices, full_like as _full_like, isfinite as _isfinite, ones_like as _ones_like
_p_arr = pval_matrix.values
_n = _p_arr.shape[0]
_iu, _ju = _triu_indices(_n, k=1)
_flat_p = _p_arr[_iu, _ju]
_finite = _isfinite(_flat_p)
_q_arr = _full_like(_flat_p, float("nan"))
if _finite.any():
    _, _q_finite, _, _ = multipletests(_flat_p[_finite], alpha=0.05, method='fdr_bh')
    _q_arr[_finite] = _q_finite
_q_matrix = _ones_like(_p_arr)
for _k, (_i_q, _j_q) in enumerate(zip(_iu, _ju)):
    _q_matrix[_i_q, _j_q] = _q_arr[_k]
    _q_matrix[_j_q, _i_q] = _q_arr[_k]
qval_matrix = DataFrame(_q_matrix, index=pval_matrix.index, columns=pval_matrix.columns)
print(f"BH-FDR survivors at q<0.05: {(_q_arr < 0.05).sum()} / {(_finite).sum()} tested pairs")
# Use q-values for downstream significance marking (replaces raw p < 0.05)
pval_matrix = qval_matrix


taxonomy = {iterativeIDs.get(k, k): v for k, v in load(open("model_inputs/taxonomy.json", 'r')).items()} # .split(".")[0]
taxonomies = {}
for col in df.columns:
    taxonomies[col] =  "|".join([v for k,v in taxonomy.get(col, "Unknown").items() if k != "Species" and v is not None])

# Create series from df.index
taxonomy_series = Series({idx: taxonomies.get(idx, f"Unknown|{idx}") for idx in df.columns})

# Debug: check detected depth
print(f"Detected depth: {max(len(t.split('|')) for t in taxonomy_series)}")
print(f"df rows: {len(df.columns)}")
print(f"taxonomy_series length: {len(taxonomy_series)}")
print(f"Sample entries:\n{taxonomy_series.head()}")

# --- Separate archaea and bacteria phyla
#NOTE There needs to be different row_colors for the different figures, hence the previous cell cannot be used
DEFAULT_COLOR = "lightgray"           
                                                                                                                                                                            
def _lighten(c, factor):                                                                                                                                                    
    h, l, s = colorsys.rgb_to_hls(*mcolors.to_rgb(c))                                                                                                                       
    return colorsys.hls_to_rgb(h, max(0.0, min(1.0, l * factor)), s)                                                                                                        
                                                                                                                                                    
proteo_base = iterativeID_color_map.get("Proteobacteria", "tab:purple")                                                                                                     
proteo_classes = sorted({                                                                                                                                                   
    str(taxonomies.get(i, "")).split("|")[2]                                                                                                                                
    for i in df.index                                                                                                                                                       
    if len(str(taxonomies.get(i, "")).split("|")) >= 3
        and str(taxonomies[i]).split("|")[1] == "Proteobacteria"                                                                                                             
})                                                                                                                                                                          
n = max(len(proteo_classes), 1)
proteo_class_color = {                                                                                                                                                      
    cls: _lighten(proteo_base, 0.6 + 0.8 * i / max(n - 1, 1))   
    for i, cls in enumerate(proteo_classes)                                                                                                                                 
}                                                               
                                                                                                                                                                            
def lookup(idx):                                                
    parts = str(taxonomies.get(idx, "")).split("|")
    if len(parts) >= 3 and parts[1] == "Proteobacteria":                                                                                                                    
        return proteo_class_color.get(parts[2], proteo_base)
    if idx in iterativeID_color_map:                                                                                                                                        
        return iterativeID_color_map[idx]                       
    elif idx in genera_color_map:                                                                                                                                           
        return genera_color_map[idx]                                                                                                                                        
    return DEFAULT_COLOR
                                                                                                                                                                            
row_colors = Series({idx: lookup(idx) for idx in corr_matrix.index}, name="Phylum")                                                                                                     
col_colors = Series({idx: lookup(idx) for idx in corr_matrix.columns}, name="Phylum")                                                                                                                          
# row_colors = Series({idx: iterativeID_color_map.get(idx, DEFAULT_COLOR) for idx in corr_matrix.index}, name=bar_label)                                                                                                                                                            
# col_colors = Series({col: iterativeID_color_map.get(col, DEFAULT_COLOR) for col in corr_matrix.columns}, name=bar_label)

clusterMap = sns.clustermap(
    corr_matrix,
    row_colors=row_colors,
    col_colors=col_colors,
    cbar_pos=None,
    cmap="coolwarm_r",
    center=0,
    figsize=(60, 70),
    dendrogram_ratio=(.1, .2)
    )
clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
# clusterMap.figure.suptitle("Abundances", fontsize=30)
clusterMap.ax_row_dendrogram.set_visible(False)
clusterMap.ax_col_dendrogram.set_visible(False)
secondary_labels = [tick.get_text() for tick in clusterMap.ax_heatmap.get_yticklabels()]
clusterMap.ax_heatmap.yaxis.set_ticks_position('left')
clusterMap.ax_heatmap.yaxis.set_label_position('left')
clusterMap.ax_heatmap.set_yticklabels(secondary_labels, rotation=0)

# Color strips sandwiched between tick labels and the heatmap                                                                                                               
hm_pos = clusterMap.ax_heatmap.get_position()                                                                                                                               
fig_w  = clusterMap.figure.get_figwidth()    # inches
fig_h  = clusterMap.figure.get_figheight()                                                                                                                                  
strip_w = 0.015   # row_colors width in figure fraction                                                                                                                      
strip_h = 0.015   # col_colors height in figure fraction         
# Row strip: flush with the left edge of the heatmap            
clusterMap.ax_row_colors.set_position([                                                                                                                                     
    hm_pos.x0 - strip_w,                                        
    hm_pos.y0,                                                                                                                                                              
    strip_w,
    hm_pos.height,                                                                                                                                                          
])                                                              
# Col strip: move from the top of the heatmap to the bottom (between heatmap and x-labels)                                                                                  
clusterMap.ax_col_colors.set_position([
    hm_pos.x0,                                                                                                                                                              
    hm_pos.y0 - strip_h,                                        
    hm_pos.width,                                                                                                                                                           
    strip_h,
])                                                                                                                                                                          
                                                                
# Push y-tick labels further left, past the row strip
y_pad = strip_w * fig_w * 72 + 15   # strip width in points + breather
clusterMap.ax_heatmap.tick_params(axis='y', pad=y_pad)                                                                                                                      
# Push x-tick labels further down, past the col strip                                                                                                                       
x_pad = strip_h * fig_h * 72 + 15                                                                                                                                           
clusterMap.ax_heatmap.tick_params(axis='x', pad=x_pad)

# cbar = clusterMap.ax_cbar
# ticks = cbar.get_yticks()
# ticks[-1] = corr_matrix.max().max()
# ticks[0] = round(corr_matrix.min().min(),2)
# cbar.set_yticklabels(ticks, fontsize=45)
# cbar.set_yticks(ticks)  # already there, but ensure
# cbar.set_xlabel("Spearman Correlation", fontsize=45, labelpad=30)
# heatmap_pos = clusterMap.ax_heatmap.get_position()
# Move colorbar to the right of the heatmap
# [left, bottom, width, height]
# clusterMap.ax_cbar.set_position([
#     heatmap_pos.x1 - 0.96,   # place it just to the right of the heatmap
#     heatmap_pos.y0 + 0.68,    # vertical position
#     0.06,                     # width of the colorbar
#     heatmap_pos.height / 5  # height of the colorbar
# ])

# axsize = 60
labelsize = 40
# clusterMap.ax_row_colors.tick_params(axis="x", labelsize=axsize)
clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=labelsize, rotation=0)
clusterMap.ax_heatmap.set_xticklabels(
    clusterMap.ax_heatmap.get_xticklabels(),
    fontsize=labelsize,
    rotation=60,
    ha='right',        # horizontal alignment anchors to the right
    rotation_mode='anchor'  # rotates around the anchor point, keeping labels flush to axis
)


# enlarge the methanogen labels
## y-axis
organisms_to_highlight = ["Methanobacterium", "Methanosarcina", "Methanobacteriaceae"]
iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", 'r'))
# ID_levels = {k.split(".")[0]: v for k, v in iterativeID_levels.items()} # 
ylabels = clusterMap.ax_heatmap.get_yticklabels()
orgs = set()
for label in ylabels:
    if any([x in label.get_text() for x in organisms_to_highlight]):  # your target labels
        orgs.add(label.get_text())
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
    text = label.get_text()
    if iterativeID_levels.get(text) == "Genus":   label.set_fontstyle("italic")
clusterMap.ax_heatmap.set_yticklabels(ylabels)

iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", 'r'))
ID_levels = {k.split(".")[0]: v for k, v in iterativeID_levels.items()}
xlabels = clusterMap.ax_heatmap.get_xticklabels()
for label in xlabels:
    if any([x in label.get_text() for x in organisms_to_highlight]):  # your target labels
        orgs.add(label.get_text())
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
    text = label.get_text()
    taxa = ID_levels.get(text)
    if taxa == "Genus":   label.set_fontstyle("italic")
clusterMap.ax_heatmap.set_xticklabels(xlabels)
    

dendrogram_row = clusterMap.dendrogram_row.reordered_ind
dendrogram_col = clusterMap.dendrogram_col.reordered_ind
one_triangle = True
if one_triangle:
    df_reordered = corr_matrix.iloc[dendrogram_row, dendrogram_col]
    mask = triu(ones_like(df_reordered, dtype=bool), k=1)  # upper triangle (above diagonal)
    mesh = clusterMap.ax_heatmap.collections[0]
    arr = mesh.get_array().reshape(df_reordered.shape)
    arr[mask] = nan
    mesh.set_array(arr.ravel())
    # clusterMap.ax_cbar.set_position([
    #     heatmap_pos.x1 - 0.2,   # place it just to the right of the heatmap
    #     heatmap_pos.y0 + 0.4,    # vertical position
    #     0.06,                     # width of the colorbar
    #     heatmap_pos.height / 5  # height of the colorbar
    # ])

for org in orgs:
    orgIx = corr_matrix.index.get_loc(org)
    # adding rows
    if orgIx not in dendrogram_row:  continue
    row_pos = dendrogram_row.index(orgIx)
    rect = patches.Rectangle(
        (0, row_pos),  # (x,y) coordinates
        len(corr_matrix.columns) if not one_triangle else row_pos+1,  # Width
        1,  # Height
        linewidth=6,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)
    # adding columns
    colIx = corr_matrix.columns.get_loc(org)
    col_pos = dendrogram_col.index(colIx)
    rect = patches.Rectangle(
        (col_pos, 0) if not one_triangle else (col_pos, len(corr_matrix.index)),  # (x,y) coordinates
        1,  # Width
        len(corr_matrix.index) if not one_triangle else -(len(corr_matrix.index)-col_pos),  # Height
        linewidth=6,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)

# highlight specific organisms
pvals_reordered = pval_matrix.iloc[dendrogram_row, dendrogram_col]
ax = clusterMap.ax_heatmap
# for i in range(pvals_reordered.shape[0]):
#     for j in range(pvals_reordered.shape[1]):
#         if pvals_reordered.iloc[i, j] < 0.05 and (not one_triangle or mask[i,j] == 0):
#             ax.add_patch(patches.Rectangle((j, i), 1, 1, fill=False, edgecolor='lightgreen', linewidth=8))
for i in range(pvals_reordered.shape[0]):
      for j in range(pvals_reordered.shape[1]):                                                                                                                               
          if pvals_reordered.iloc[i, j] < 0.05 and (not one_triangle or mask[i, j] == 0):
              ax.text(j + 0.5, i + 0.5, "*",                                                                                                                                  
                      color="lightgreen",                                                                                                                                     
                      fontsize=60,                # tune for cell size                                                                                                        
                      fontweight="bold",                                                                                                                                      
                      ha="center", va="center")

clusterMap.ax_heatmap.set_xlabel("Member ASVs", fontsize=50)
clusterMap.ax_heatmap.set_ylabel("Member ASVs", fontsize=50)

# --- Taxonomy legend (row/col color strips) -------------------------------
# display(taxa_color_map)
# `order_color_map`, `archaea_phyla`, `bacteria_phyla`, and `level` must be                                                                                                
# defined in scope (same names used by the row_colors / col_colors construction).                                                                                          
fig_width = clusterMap.figure.get_figwidth()  # inches; same role as `width` previously  
phylum_color = {}                                                                                                                                                           
for idx in corr_matrix.index:   
    parts = str(taxonomies.get(idx, "")).split("|")                                                                                                                         
    if len(parts) >= 2 and parts[1] not in ("None", "", "Unknown", "nan"):
        phylum_color.setdefault(parts[1], row_colors.get(idx))
# Phyla that actually appear in the rendered heatmap                                                                                                                        
shown_phyla = set()                                             
for idx in corr_matrix.index:
    parts = str(taxonomies.get(idx, "")).split("|")                                                                                                                         
    if len(parts) >= 2 and parts[1] not in ("None", "", "Unknown", "nan"):
        shown_phyla.add(parts[1])                                                                                                                                           
                                                                                                                                                                            
archaea_markers = ("archaeo", "euryarchaeota", "crenarchaeota", "thaumarchaeota",                                                                                           
                    "halobacterota", "methanobacteriota", "micrarchaeota", "nanoarchaeota")                                                                                  
def is_archaea(p): return any(m in p.lower() for m in archaea_markers)      
archaea_phyla  = sorted(p for p in shown_phyla if is_archaea(p))                                                                                                            
bacteria_phyla = sorted(p for p in shown_phyla if not is_archaea(p))

handle = [] 

def header_patch(title):
    """Section header: invisible swatch with a bold label."""
    return patches.Patch(color="none", label=fr"$\bf{{{title}}}$")

if len(archaea_phyla) > 0:                                                                                 
    archaea_patches  = [patches.Patch(color=phylum_color.get(p, DEFAULT_COLOR), label=p)                                                                                                       
                        for p in archaea_phyla] 
    handle = [header_patch("Archaea")]  + archaea_patches             
if len(bacteria_phyla) > 0:                                                                                           
    bacteria_patches = [patches.Patch(color=phylum_color.get(p, DEFAULT_COLOR), label=p)                                                                                                       
                        for p in bacteria_phyla]     
    handle += [header_patch("Bacteria")] + bacteria_patches                                                                                                 
legend_handles = (handle)
                                                                                                                                                                            
clusterMap.ax_heatmap.legend(                                                                                                                                              
    handles=legend_handles,
    title=level,                                                                                                                                            
    title_fontsize=8 * (fig_width / 10),                        
    loc="lower left",                                                                                                                                                      
    bbox_to_anchor=(0.5, 0.6),                                                                                                                                           
    fontsize=7 * (fig_width / 10),                                                                                                                                         
    frameon=True,                                                                                                                                                          
)

# clusterMap.figure.suptitle("Abundance correlational matrix", fontsize=100, y=1.01)
# clusterMap.figure.tight_layout()
clusterMap.figure.savefig(f"abundance_heatmaps/abundance_correlatons{'_one_triangle' if one_triangle else ''}.png",
    dpi=300, bbox_inches='tight')

In [ ]:
pval_matrix[pval_matrix < 0.05]

In [ ]:
# sig_mask is a boolean DataFrame (True = p < threshold)
corr_masked = corr_df.where(sig_mask, 0)
km = KMeans(n_clusters=best_k, n_init=20, random_state=42)
cluster_labels = pd.Series(km.fit_predict(corr_masked), index=corr_df.index)

#### methanogen correlations

In [ ]:
from matplotlib import colors, pyplot, patches
from matplotlib.patches import Patch
from pandas import Series, DataFrame
from json import load
from numpy import ones
import seaborn as sns

days = {
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}


from scipy.stats import spearmanr
def corr_pvalues(df):
    n = df.shape[1]
    pvals = DataFrame(ones((n, n)), index=df.columns, columns=df.columns)
    for i in range(n):
        for j in range(i+1, n):
            _, p = spearmanr(df.iloc[:, i], df.iloc[:, j])
            pvals.iloc[i, j] = p
            pvals.iloc[j, i] = p
    return pvals

iterativeIDs = load(open("model_inputs/iterativeIDs.json", 'r'))
df = DataFrame(load(open("model_inputs/abundances.json", 'r'))).T
df.drop(df.index.difference(days.keys()), inplace=True)
df.columns = [iterativeIDs.get(ID, ID) for ID in df.columns] #columns
df = df.T.groupby(df.columns).sum().T

taxonomy = {iterativeIDs.get(k, k): v for k, v in load(open("model_inputs/taxonomy.json", 'r')).items()}
taxonomies = {col: "|".join([v for k,v in taxonomy.get(col, {}).items() if k != "Species" and v is not None])
                for col in df.columns}
taxonomy_series = Series(taxonomies)

df.drop([col for col in df.columns if taxonomies.get(col, "|").split("|")[0] != "Archaea" or "midas" in col], axis=1, inplace=True) #"Methano" not in col], axis=1, inplace=True)
corr_matrix = df.corr("spearman").dropna(axis=1, how='all').dropna(axis=0, how='all').fillna(0)
display(corr_matrix)
pval_matrix = corr_pvalues(df).dropna(axis=1, how='all').dropna(axis=0, how='all').fillna(0)


# === BH-FDR on upper-triangle p-values to derive q-value matrix ===
from statsmodels.stats.multitest import multipletests
from numpy import triu_indices as _triu_indices, full_like as _full_like, isfinite as _isfinite, ones_like as _ones_like
_p_arr = pval_matrix.values
_n = _p_arr.shape[0]
_iu, _ju = _triu_indices(_n, k=1)
_flat_p = _p_arr[_iu, _ju]
_finite = _isfinite(_flat_p)
_q_arr = _full_like(_flat_p, float("nan"))
if _finite.any():
    _, _q_finite, _, _ = multipletests(_flat_p[_finite], alpha=0.05, method='fdr_bh')
    _q_arr[_finite] = _q_finite
_q_matrix = _ones_like(_p_arr)
for _k, (_i_q, _j_q) in enumerate(zip(_iu, _ju)):
    _q_matrix[_i_q, _j_q] = _q_arr[_k]
    _q_matrix[_j_q, _i_q] = _q_arr[_k]
pval_matrix = DataFrame(_q_matrix, index=pval_matrix.index, columns=pval_matrix.columns)
print(f"BH-FDR survivors at q<0.05: {(_q_arr < 0.05).sum()} / {(_finite).sum()} tested pairs")
pval_matrix = pval_matrix[corr_matrix.columns]
pval_matrix = pval_matrix.loc[corr_matrix.index]
display(pval_matrix)

# tax_series = Series(taxonomies)
# print("Taxonomy index matches df.index:", tax_series.index.equals(df.index))
# print("Taxonomy index order matches:", list(tax_series.index) == list(df.index))

# display(df)
# display(df.to_numpy())
# Create series from df.index
# taxonomy_series = Series({idx: taxonomies.get(idx, f"Unknown|{idx}") for idx in df.columns})

# Debug: check detected depth
# print(f"Detected depth: {max(len(t.split('|')) for t in taxonomy_series)}")
print(f"df rows: {len(df.columns)}")
print(f"taxonomy_series length: {len(taxonomy_series)}")
print(f"Sample entries:\n{taxonomy_series.head()}")

# max_abundance = max(list(df.to_numpy().flatten()))

# def create_taxonomy_colors(taxonomy_series, df_index):
#     """Create row_colors showing taxonomy at each level."""
#     taxonomy_series = taxonomy_series.reindex(df_index)
#     parsed = taxonomy_series.str.split(r"[;|,]", regex=True)
#     depth = max(len(p) for p in parsed)
    
#     level_names = ['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'][:depth]
#     levels_df = DataFrame(index=df_index)
    
#     for i, name in enumerate(level_names):
#         levels_df[name] = parsed.apply(lambda x: x[i] if i < len(x) else "")
    
#     # Create colors
#     import matplotlib.pyplot as plt
#     row_colors = DataFrame(index=df_index)
#     for col in levels_df.columns:
#         unique_vals = levels_df[col].unique()
#         cmap = plt.cm.get_cmap('tab20', len(unique_vals))
#         color_map = {v: cmap(i) for i, v in enumerate(unique_vals)}
#         row_colors[col] = levels_df[col].map(color_map)
    
#     return row_colors

# cmap = LinearSegmentedColormap.from_list("NewMap", ["red", "skyblue"])
# display(df)
# clustermap = ClusterHeatmap(df, level, ) # Series(dict(Counter(taxonomies))))
clusterMap = sns.clustermap(corr_matrix,
                            # row_colors=row_colors,
                            cmap="coolwarm_r",
                            center=0,
                            # vmin=0,
                            figsize=(60, 60),
                            # vmax=max(list(df.to_numpy().flatten())),
                            # col_cluster=False,
                            # row_linkage=taxonomy_linkage(taxonomy_series),
                            # col_linkage=taxonomy_linkage(taxonomy_series),
                            # row_colors=create_taxonomy_colors(taxonomy_series, df.columns)[['Kingdom']],
                            dendrogram_ratio=(.1, .2))
clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
# clusterMap.figure.suptitle("Abundances", fontsize=30)

# highlight specific organisms
dendrogram_row = clusterMap.dendrogram_row.reordered_ind
dendrogram_col = clusterMap.dendrogram_col.reordered_ind
# print(dendrogram_row)
# print(dendrogram_col)
organisms_to_highlight = ["Methanobacterium.2", "Methanobacteriaceae.1", "Methanobacterium.1"]
for org in organisms_to_highlight:
    orgIx = corr_matrix.index.get_loc(org)
    # adding rows
    if orgIx not in dendrogram_row:  continue
    row_pos = dendrogram_row.index(orgIx)
    rect = patches.Rectangle(
        (0, row_pos),  # (x,y) coordinates
        len(corr_matrix.columns),  # Width
        1,  # Height
        linewidth=6,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)
    # adding columns
    colIx = corr_matrix.columns.get_loc(org)
    col_pos = dendrogram_col.index(colIx)
    rect = patches.Rectangle(
        (col_pos, 0),  # (x,y) coordinates
        1,  # Width
        len(corr_matrix.index),  # Height
        linewidth=6,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)

pvals_reordered = pval_matrix.iloc[dendrogram_row, dendrogram_col]
ax = clusterMap.ax_heatmap
for i in range(pvals_reordered.shape[0]):
    for j in range(pvals_reordered.shape[1]):
        if pvals_reordered.iloc[i, j] < 0.05:
            ax.add_patch(patches.Rectangle((j, i), 1, 1, fill=False, edgecolor='lightgreen', linewidth=8))

cbar = clusterMap.ax_cbar
ticks = cbar.get_yticks()
ticks[-1] = round(corr_matrix.max().max())
ticks[0] = round(corr_matrix.min().min(),2)
cbar.set_yticklabels(ticks, fontsize=35)
cbar.set_yticks(ticks)  # already there, but ensure
cbar.set_xlabel("Spearman", fontsize=40, labelpad=20)
# axsize = 60
labelsize = 40
# clusterMap.ax_row_colors.tick_params(axis="x", labelsize=axsize)
clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=labelsize, rotation=0)
clusterMap.ax_heatmap.set_xticklabels(
    clusterMap.ax_heatmap.get_xticklabels(),
    fontsize=labelsize,
    rotation=60,
    ha='right',        # horizontal alignment anchors to the right
    rotation_mode='anchor'  # rotates around the anchor point, keeping labels flush to axis
)

# enlarge the notable organisms labels
# y-axis
ylabels = clusterMap.ax_heatmap.get_yticklabels()
for label in ylabels:
    if label.get_text() in organisms_to_highlight:  # your target labels
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
clusterMap.ax_heatmap.set_yticklabels(ylabels)

xlabels = clusterMap.ax_heatmap.get_xticklabels()
for label in xlabels:
    if label.get_text() in organisms_to_highlight:  # your target labels
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
clusterMap.ax_heatmap.set_xticklabels(xlabels)

clusterMap.ax_row_dendrogram.set_visible(False)
clusterMap.ax_col_dendrogram.set_visible(False)
heatmap_pos = clusterMap.ax_heatmap.get_position()
clusterMap.ax_cbar.set_position([
    heatmap_pos.x1 - 0.96,   # place it just to the right of the heatmap
    heatmap_pos.y0 + 0.5,    # vertical position
    0.06,                     # width of the colorbar
    heatmap_pos.height / 5  # height of the colorbar
])

clusterMap.ax_heatmap.set_xlabel("Member ASVs", fontsize=50)
clusterMap.ax_heatmap.set_ylabel("Member ASVs", fontsize=50)
# clusterMap.figure.suptitle("Abundance correlational matrix", fontsize=100, y=1.01)
# clusterMap.figure.tight_layout()
clusterMap.figure.savefig("abundance_heatmaps/methanogens_abundance_correlatons.png", bbox_inches='tight', dpi=300)

In [ ]:
from matplotlib import colors, pyplot, patches
from matplotlib.patches import Patch
from pandas import Series, DataFrame
from json import load
from numpy import log10, inf, nan, ones, triu, ones_like
import seaborn as sns

days = {
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}


from scipy.stats import spearmanr
def corr_pvalues(df):
    n = df.shape[1]
    pvals = DataFrame(ones((n, n)), index=df.columns, columns=df.columns)
    for i in range(n):
        for j in range(i+1, n):
            _, p = spearmanr(df.iloc[:, i], df.iloc[:, j])
            pvals.iloc[i, j] = p
            pvals.iloc[j, i] = p
    return pvals


nonzero_per_day = load(open("model_inputs/nonzero_per_day.json", "r"))
top_per_day = {}
all_orgs = {}
topNum = 10
for day, org_dict in nonzero_per_day.items():
    orgs = dict(list(org_dict.items())[:topNum])
    top_per_day[day] = orgs #{k:log10(v) for k,v in orgs.items()}
    all_orgs.update(orgs)
df = DataFrame(top_per_day).T
df = df.astype(float).replace([inf, -inf], nan)#.fillna(0)
df = df.loc[:, df.notna().sum() >= 5]
display(df)

corr_matrix = df.corr("spearman").dropna(axis=1, how='all').dropna(axis=0, how='all').fillna(0)
# corr_matrix = (corr_matrix > ).astype(int)
display(corr_matrix)
pval_matrix = corr_pvalues(df).dropna(axis=1, how='all').dropna(axis=0, how='all').fillna(0)
pval_matrix = pval_matrix[corr_matrix.columns]
pval_matrix = pval_matrix.loc[corr_matrix.index]
display(pval_matrix)

clusterMap = sns.clustermap(corr_matrix,
                            # row_colors=row_colors,
                            cmap="coolwarm_r",
                            center=0,
                            # vmin=0,
                            figsize=(60, 60),
                            # vmax=max(list(df.to_numpy().flatten())),
                            col_cluster=True,
                            row_cluster=True,
                            # row_linkage=taxonomy_linkage(taxonomy_series),
                            # col_linkage=taxonomy_linkage(taxonomy_series),
                            # row_colors=create_taxonomy_colors(taxonomy_series, df.columns)[['Phylum']],
                            dendrogram_ratio=(.1, .2))
clusterMap.figure.subplots_adjust(bottom=0.15, top=0.95)  # Adjust these values as needed to fit labels
# clusterMap.figure.suptitle("Abundances", fontsize=30)

# highlight specific organisms
dendrogram_row = clusterMap.dendrogram_row.reordered_ind
dendrogram_col = clusterMap.dendrogram_col.reordered_ind
one_triangle = True
if one_triangle:
    df_reordered = corr_matrix.iloc[dendrogram_row, dendrogram_col]
    mask = triu(ones_like(df_reordered, dtype=bool), k=1)  # upper triangle (above diagonal)
    mesh = clusterMap.ax_heatmap.collections[0]
    arr = mesh.get_array().reshape(df_reordered.shape)
    arr[mask] = nan
    mesh.set_array(arr.ravel())
    
# print(dendrogram_row)
# print(dendrogram_col)
organisms_to_highlight = ["Methanobacterium.2", "Methanobacteriaceae.1", "Methanobacterium.1"]
for org in organisms_to_highlight:
    orgIx = corr_matrix.index.get_loc(org)
    # adding rows
    if orgIx not in dendrogram_row:  continue
    row_pos = dendrogram_row.index(orgIx)
    rect = patches.Rectangle(
        (0, row_pos),  # (x,y) coordinates
        len(corr_matrix.columns) if not one_triangle else row_pos+1,  # Width
        1,  # Height
        linewidth=16,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)
    # adding columns
    colIx = corr_matrix.columns.get_loc(org)
    col_pos = dendrogram_col.index(colIx)
    rect = patches.Rectangle(
        (col_pos, 0) if not one_triangle else (col_pos, len(corr_matrix.index)),  # (x,y) coordinates
        1,  # Width
        len(corr_matrix.index) if not one_triangle else -(len(corr_matrix.index)-col_pos),  # Height
        linewidth=16,  # Stroke width
        edgecolor='black',  # Stroke color
        facecolor='none',  # No fill color
        # transform=clusterMap.ax_heatmap.get_yaxis_transform(),
        clip_on=False
    )
    clusterMap.ax_heatmap.add_patch(rect)

# pvals_reordered = pval_matrix.iloc[dendrogram_row, dendrogram_col]
# ax = clusterMap.ax_heatmap
# for i in range(pvals_reordered.shape[0]):
#     for j in range(pvals_reordered.shape[1]):
#         if pvals_reordered.iloc[i, j] < 0.05:
#             ax.add_patch(patches.Rectangle((j, i), 1, 1, fill=False, edgecolor='lightgreen', linewidth=15))

cbar = clusterMap.ax_cbar
ticks = cbar.get_yticks()
ticks[-1] = round(corr_matrix.max().max())
ticks[0] = round(corr_matrix.min().min(),2)
cbar.set_yticklabels(ticks, fontsize=50)
cbar.set_yticks(ticks)  # already there, but ensure
# cbar.set_xlabel(r"Spearman $\rho$", fontsize=60, labelpad=20)
cbar.set_ylabel(r"Spearman $\rho$", fontsize=60, labelpad=15, rotation=90)
# axsize = 60
labelsize = 60
# clusterMap.ax_row_colors.tick_params(axis="x", labelsize=axsize)
clusterMap.ax_heatmap.set_yticklabels(clusterMap.ax_heatmap.get_yticklabels(), fontsize=labelsize, rotation=0)
clusterMap.ax_heatmap.set_xticklabels(
    clusterMap.ax_heatmap.get_xticklabels(),
    fontsize=labelsize,
    rotation=60,
    ha='right',        # horizontal alignment anchors to the right
    rotation_mode='anchor'  # rotates around the anchor point, keeping labels flush to axis
)

## add values text to the elements
for i in range(clusterMap.data2d.shape[0]):
    for j in range(clusterMap.data2d.shape[1]):
        if mask[i,j] == 1:  continue
        value = clusterMap.data2d.iloc[i, j]
        if value == 0:  continue
        color = "black" if abs(value) < 0.7 else "white"
        clusterMap.ax_heatmap.text(j+0.5, i+0.5, f'{value:.2f}', ha='center',
                                   va='center', color=color, fontsize=60)

# enlarge the notable organisms labels
# y-axis
iterativeID_levels = load(open("model_inputs/iterativeID_levels.json", 'r'))
ID_levels = {k.split(".")[0]: v for k, v in iterativeID_levels.items()}
clusterMap.ax_heatmap.yaxis.set_ticks_position('left')
clusterMap.ax_heatmap.yaxis.set_label_position('left')
ylabels = clusterMap.ax_heatmap.get_yticklabels()
for label in ylabels:
    if label.get_text() in organisms_to_highlight:  # your target labels
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
    text = label.get_text().split(".")[0]
    taxa = ID_levels.get(text)
    if taxa == "Genus":   label.set_fontstyle("italic")
    # label.set_rotation(0)
clusterMap.ax_heatmap.set_yticklabels(ylabels, rotation=0)

xlabels = clusterMap.ax_heatmap.get_xticklabels()
for label in xlabels:
    if label.get_text() in organisms_to_highlight:  # your target labels
        label.set_fontsize(labelsize*1.2)
        label.set_fontweight('bold')  # optional
    text = label.get_text().split(".")[0]
    taxa = ID_levels.get(text)
    if taxa == "Genus":   label.set_fontstyle("italic")
    # label.set_rotation(0)
clusterMap.ax_heatmap.set_xticklabels(xlabels)

clusterMap.ax_row_dendrogram.set_visible(False)
clusterMap.ax_col_dendrogram.set_visible(False)
heatmap_pos = clusterMap.ax_heatmap.get_position()
clusterMap.ax_cbar.set_position([
    heatmap_pos.x1 - 1.01,   # place it just to the right of the heatmap
    heatmap_pos.y0 + 0.5,    # vertical position
    0.06,                     # width of the colorbar
    heatmap_pos.height / 5  # height of the colorbar
])
if one_triangle:
    clusterMap.ax_cbar.set_position([
        heatmap_pos.x1 - 0.3,   # place it just to the right of the heatmap
        heatmap_pos.y0 + 0.4,    # vertical position
        0.06,                     # width of the colorbar
        heatmap_pos.height / 5  # height of the colorbar
    ])


# highlight specific organisms
# pvals_reordered = pval_matrix.iloc[dendrogram_row, dendrogram_col]
# ax = clusterMap.ax_heatmap
# for i in range(pvals_reordered.shape[0]):
#     for j in range(pvals_reordered.shape[1]):
#         if pvals_reordered.iloc[i, j] < 0.05 and (not one_triangle or mask[i,j] == 0):
#             ax.add_patch(patches.Rectangle((j, i), 1, 1, fill=False, edgecolor='lightgreen', linewidth=8))


clusterMap.ax_heatmap.set_xlabel("ASVs", fontsize=100, labelpad=20)
clusterMap.ax_heatmap.set_ylabel("ASVs", fontsize=100, labelpad=20)
# clusterMap.figure.suptitle("Abundance correlational matrix", fontsize=100, y=1.01)
# clusterMap.figure.tight_layout()
clusterMap.figure.savefig(f"abundance_heatmaps/Top_{topNum}_ASVs_abundance_correlation{'_one_triangle' if one_triangle else ''}.png", bbox_inches='tight', dpi=300)

# co-occurrence figure

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import math
import numpy as np
from pandas import DataFrame
from itertools import combinations
from collections import defaultdict
from scipy.stats import spearmanr
from json import load, dump
from statsmodels.stats.multitest import multipletests

days = {
    "H": 140,
    "Ha": 147,
    "I": 149,
    "Ia": 153,
    "J": 161,
    "Ja": 168,
    "Jb": 174,
    "K": 182,
    "L": 189,
    "M": 210,
    "N": 238,
    "Na": 262,
    "P": 283,
    "Q": 300,
    "Qa": 315,
}

# --- Order -> color mapping
iterativeIDs = load(open("model_inputs/iterativeIDs.json"))
taxonomy = load(open("model_inputs/taxonomy.json"))
iterativeID_taxonomy = {iterativeIDs.get(ID, ID): taxonomy.get(ID, "Unknown") for ID in iterativeIDs}
level = "Phylum"
iterativeID_level = {ID: content.get(level, "Unknown") for ID, content in iterativeID_taxonomy.items()}
genera_level = False
if genera_level:
    iterativeID_level = iterativeID_level = {ID.split(".")[0]: k for ID, k in iterativeID_level.items()}

# iterativeID_taxonomy = {iterativeIDs.get(hash, hash): taxonomy.get(hash, "Unknown") for hash in iterativeIDs}
# level = "Phylum"
# iterativeID_level = {ID: content.get(level, "Unknown") for ID, content in iterativeID_taxonomy.items()}
# genera_level = True
# if not genera_level:
#     iterativeID_level = {ID.split(".")[0]: k for ID, k in iterativeID_level.items()}

none_keys = [k for k, v in iterativeID_level.items() if v is None]
print("None taxonomy entries:", none_keys)
# palette = plt.cm.tab20.colors
# --- Separate archaea and bacteria phyla
archaea_phyla = sorted({v for k, v in iterativeID_level.items() 
                        if v is not None and (
                            "archaeo" in v.lower() or 
                            any(a in v.lower() for a in [
                                "euryarchaeota", "crenarchaeota",
                                "thaumarchaeota", "candidatus thermoplasmatota",
                                "halobacterota", "methanobacteriota",
                                "micrarchaeota", "nanoarchaeota"
                            ])
                        )})
bacteria_phyla = sorted({v for k, v in iterativeID_level.items() if v is not None and v not in archaea_phyla})

# Archaea: red end of spectrum (0.0 - 0.15 in turbo)
n_archaea = len(archaea_phyla)
archaea_colors = [plt.cm.turbo(i / max(n_archaea, 1) * 0.15) 
                  for i in range(n_archaea)]

# Bacteria: rest of spectrum (0.2 - 1.0 in turbo)
n_bacteria = len(bacteria_phyla)
bacteria_colors = [plt.cm.turbo(0.2 + i / max(n_bacteria, 1) * 0.8) 
                   for i in range(n_bacteria)]

# Combine into one color map
taxa_color_map = {}
for phylum, color in zip(archaea_phyla, archaea_colors):
    taxa_color_map[phylum] = color
for phylum, color in zip(bacteria_phyla, bacteria_colors):
    taxa_color_map[phylum] = color
dump(taxa_color_map, open(f"{level}_color_map.json", "w"))
iterativeID_color_map = {ID: taxa_color_map[phylum] for ID, phylum in iterativeID_level.items() if phylum}
dump(iterativeID_color_map, open(f"iterativeID_color_map.json", "w"))

# --- Relative abundance DataFrame
df = DataFrame(load(open("model_inputs/abundances.json", 'r'))).T
df.drop(df.index.difference(days.keys()), inplace=True)
df = df.loc[:, (df.fillna(0) > 0).sum() >= 3]
df.columns = [iterativeIDs.get(ID, ID) for ID in df.columns]
if genera_level:
    df.columns = [col.split(".")[0] for col in df.columns] #columns
    df = df.T.groupby(df.columns).sum().T
# to_drop = [col for col in df.columns if df[col].max() < 0.001]
# print("Dropping columns:", to_drop)
# df.drop(to_drop, axis=1, inplace=True)
display(df)
# df = df.fillna(0)
relative_abundance = df.div(df.sum(axis=1), axis=0)
mean_rel_abund = relative_abundance.mean(axis=0)

# presence = (df > 0).astype(int)
# cooccurrence = defaultdict(int)
# presence_count = defaultdict(int)
# for col in presence.columns:
#     presence_count[col] = presence[col].sum()
# for sample in presence.itertuples(index=False):
#     present = [col for col, val in zip(presence.columns, sample) if val]
#     for pair in combinations(sorted(present), 2):
#         cooccurrence[pair] += 1

# --- 2. Build graph: edge exists if co-occurrence >= threshold
# min_cooccurrence = 1   # was 3 — require more shared samples
# min_rho = 0.6          # 0.3, only keep meaningful correlations
# significant_edges = 0
# pvals,  = []
# min_p_value = 0.05
# for (a, b), count in cooccurrence.items():
#     # if count >= min_cooccurrence:
#     rho, p_value = spearmanr(df[a], df[b])
#     if not np.isnan(rho) and p_value <= min_p_value: #abs(rho) >= min_rho and p_value <= min_p_value:
#         G.add_edge(a, b, weight=abs(rho), rho=rho, cooccurrence=count)
#         significant_edges += 1
#         pvals.append(p_value)
#         if abs(rho) < 0.7:
#             raise ValueError(f"rho is too low: {rho}")
# print("significant edges:", significant_edges)

presence = (df > 0).astype(int)
cooccurrence = defaultdict(int)
for sample in presence.itertuples(index=False):
    present = [col for col, val in zip(presence.columns, sample) if val]
    for pair in combinations(sorted(present), 2):
        cooccurrence[pair] += 1

# min_cooccurrence = 3   # pair must co-occur in at least this many samples to be tested

# --- 2. Pass 1: run Spearman on every qualifying pair, keep results in memory
pair_data = []   # each entry: (a, b, rho, p_value, count)
for (a, b), count in cooccurrence.items():
    # if count < min_cooccurrence:
    #     continue
    rho, p_value = spearmanr(df[a], df[b])
    if np.isnan(rho): continue
    pair_data.append((a, b, rho, p_value, count))

# --- 3. Pass 2: BH-FDR correction across all tests actually run
pvals = np.array([t[3] for t in pair_data])
reject, qvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

# --- 4. Pass 3: build the graph from surviving pairs
G = nx.Graph()
# min_rho = 0.6   # optional extra magnitude floor; set to 0.0 to disable
for (a, b, rho, p, count), q, keep in zip(pair_data, qvals, reject):
    if keep: # and abs(rho) >= min_rho:
        G.add_edge(a, b,weight=abs(rho), rho=rho, pvalue=p, qvalue=q, cooccurrence=count)

print(f"Tests run: {len(pair_data)}")
print(f"FDR-significant pairs (q < 0.05): {int(reject.sum())}")
print(f"Edges after FDR correction: {G.number_of_edges()}")
print(f"Nodes in graph: {G.number_of_nodes()}")



# --- 3. Layout
print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
print("max cooccurrence:", max(cooccurrence.values()) if cooccurrence else 0)
# --- Community-aware two-level layout ---
# 1. Detect Louvain communities up-front (so layout can respect them)
try:
    _layout_comms = nx.community.louvain_communities(
        G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["rho"] > 0]).copy(),
        weight="weight", resolution=1.0, seed=42)
except AttributeError:
    _layout_comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
_node_module = {n: i for i, c in enumerate(_layout_comms) for n in c}

# 2. Build a "meta" graph (one super-node per community, edges = aggregated inter-community weight)
G_meta = nx.Graph()
for _i in range(len(_layout_comms)):
    G_meta.add_node(_i, size=len(_layout_comms[_i]))
for u, v, d in G.edges(data=True):
    cu, cv = _node_module.get(u), _node_module.get(v)
    if cu is None or cv is None or cu == cv:
        continue
    if G_meta.has_edge(cu, cv):
        G_meta[cu][cv]["weight"] += d["weight"]
    else:
        G_meta.add_edge(cu, cv, weight=d["weight"])
_meta_pos = nx.spring_layout(G_meta, seed=42, k=3.0, iterations=200, scale=2.5)

# 3. Layout each community's interior locally, then translate to its centroid
pos = {}
for _i, _comm in enumerate(_layout_comms):
    _members = list(_comm)
    if len(_members) == 1:
        pos[_members[0]] = np.array(_meta_pos[_i])
        continue
    _G_sub = G.subgraph(_members).copy()
    # local layout — small `k` keeps members close to their centroid
    _local = nx.spring_layout(_G_sub, seed=42, k=0.6, iterations=100, scale=0.6)
    _centroid = np.array(_meta_pos[_i])
    for _n, _p in _local.items():
        pos[_n] = np.array(_p) + _centroid
# Catch any nodes Louvain missed (shouldn't happen, but safety):
for _n in G.nodes():
    if _n not in pos:
        pos[_n] = np.array([0.0, 0.0])   # spread further   # spread further (was k=2.5, iter=200)   # expanded layout (was k=0.3, iter=100) #, k=0.9) #, k=2) #, k=.8 with rho=0.8) # k=1.5 for the smaller map
# Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
# H = Gc.copy()
# for u, v, d in H.edges(data=True):
#     d['weight'] = 1.0 / abs(d['weight'])        # or:
#     d['weight'] = -np.log(abs(d['weight']) + 1e-6)

# init = nx.spectral_layout(H, weight='weight')
# pos = nx.kamada_kawai_layout(H, pos=init, weight='weight')

edges = G.edges(data=True)
rho_values = [d["rho"] for _, _, d in edges]
edge_widths = [3 * d["weight"] for _, _, d in edges]
# Print degree sorted descending
print(sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10])

# Diverging colormap centered at 0
norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
cmap = plt.get_cmap("coolwarm_r") #cmap = cm.RdBu
edge_colors = [cmap(norm(r)) for r in rho_values]

# --- 6. Draw — define width/height FIRST
width = 40
height = 30
fig, ax = plt.subplots(figsize=(width, height))

# --- 5. Node styling — scale AFTER width is defined
scale = 5000 * (width / 10) * 2
compressor = np.sqrt

# --- Color palette — expanded to avoid repeats
# n_orders = len(set(iterativeID_level.values()))
# palette = [plt.cm.turbo(i / n_orders) for i in range(n_orders)]
# order_color_map = {
#     order: palette[i]
#     for i, order in enumerate(sorted(v for v in set(iterativeID_level.values()) if v is not None))
# }

node_sizes = [scale * compressor(mean_rel_abund.get(n, 0)) for n in G.nodes()]
node_colors = [taxa_color_map.get(iterativeID_level.get(n, "Unknown"), "lightgray") for n in G.nodes()]
print("unique node colors:", len(set(node_colors)))

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors, alpha=0.85, ax=ax)

significantly_connected_organisms = list(pos.keys())
print(significantly_connected_organisms)
dump(significantly_connected_organisms, open("significantly_connected_organisms.json", "w"))
LABEL_MIN_ABUND = 0.002   # 0.2%                                                                                                                                               
for n, (x, y) in pos.items():                                                                                                                                               
    abund = mean_rel_abund.get(n, 0)
    if abund < LABEL_MIN_ABUND:                                                                                                                                             
        continue     
    font_size = 6 + 14 * compressor(abund) / compressor(mean_rel_abund.max()) * (width / 10)
    font_size = max(5, min(font_size, 48))                                                                                                                                  
    ax.text(x, y, str(n), fontsize=font_size, color="black", fontweight="bold", ha="center", va="center")

# --- 7. Colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label(r"Spearman $\rho$", fontsize=10 * (width / 10))
cbar.ax.tick_params(labelsize=8 * (width / 10), length=8, width=2)

# --- 9. Size legend FIRST — pin it with add_artist
# for abund, label in [(0.002, "0.2%"), (0.02, "2%"), (0.20, "20%")]:                                                                                                        
#     ax.scatter([], [], s=scale * compressor(abund), color="slategray", alpha=0.9, label=label)
# size_legend = ax.legend(title="Mean rel. abundance",                                                                                                                        
#                         title_fontsize=6 * (width / 10),   # was 8
#                         labelspacing=4.0,                                                                                               
#                         handletextpad=1.5,                  # was 2.5
#                         borderpad=1.0,                      # was 1.5                                                                                                       
#                         loc="upper left",                       
#                         bbox_to_anchor=(0.95, 1.0),                                                                                                                          
#                         frameon=True, fontsize=5 * (width / 10))  # was 7                                                                                                   
# ax.add_artist(size_legend)  # pin size legend before overwriting with taxonomy
# Custom size legend with spacing proportional to circle diameters
legend_entries = [(0.002, "0.2%"), (0.02, "2%"), (0.20, "20%")]                                                                                                             
sizes_pt2    = [scale * compressor(a) for a, _ in legend_entries]   # full-scale, same formula as nodes                                                                     
diameters_pt = [2 * np.sqrt(s / np.pi) for s in sizes_pt2]      
breather_pt = 8                                                                                                                                                             
offsets_pt  = [0.0]
for i in range(1, len(legend_entries)):                                                                                                                                     
    offsets_pt.append(offsets_pt[-1] + (diameters_pt[i-1] + diameters_pt[i]) / 2 + breather_pt)                                                                                
fig        = ax.figure                                                                                                                                                      
fig_h_pts  = fig.get_figheight() * 72       # fig height in points
top_y      = 0.9                            # top-of-legend figure fraction                                                                                                
circle_x   = 0.85                            # figure fraction for the circles
label_x    = 0.88                           # figure fraction for the labels                                                                                               
                                                                
# Title                                                                                                                                                                     
ax.text(circle_x, top_y + 0.025, "Mean rel. abundance",         
        transform=fig.transFigure, va='bottom', fontweight='bold',                                                                                                          
        fontsize=6 * (width/10), clip_on=False)
                                                                                                                                                                            
# Circles + labels, stacked top-down                            
for (abund, label), s, off in zip(legend_entries, sizes_pt2, offsets_pt):                                                                                                   
    y = top_y - off / fig_h_pts                                                                                                                                             
    ax.scatter([circle_x], [y], s=s, color="slategray", alpha=0.9,
                transform=fig.transFigure, clip_on=False)                                                                                                                    
    ax.text(label_x, y, label,                                  
            transform=fig.transFigure, va='center',                                                                                                                         
            fontsize=5 * (width/10), clip_on=False)

# --- 8. Taxonomy legend with sub-titles LAST
archaea_patches = [mpatches.Patch(color=taxa_color_map[p], label=p) for p in archaea_phyla if p in taxa_color_map]
bacteria_patches = [mpatches.Patch(color=taxa_color_map[p], label=p) for p in bacteria_phyla if p in taxa_color_map]

# Blank patch used as a section header — invisible box, bold label
def header_patch(title):
    return mpatches.Patch(color="none", label=f"$\\bf{{{title}}}$")

legend_handles = (
    [header_patch("Archaea")] +
    archaea_patches +
    [header_patch("Bacteria")] +
    bacteria_patches
)

ax.legend(handles=legend_handles, title="Taxonomic "+level,
          title_fontsize=8 * (width / 10),
          loc="lower left", bbox_to_anchor=(-0.24, 0.1),
          fontsize=7 * (width / 10), frameon=True)

ax.axis("off")
plt.tight_layout()

# Annotate modularity (computed in metrics cell — reproduce in-line for figure self-containment)
try:
    _comms_for_label = nx.community.louvain_communities(
        G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["rho"] > 0]).copy(),
        weight="weight", resolution=1.0, seed=42)
    _Q_for_label = nx.community.modularity(
        G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["rho"] > 0]).copy(),
        _comms_for_label, weight="weight")
except AttributeError:
    _comms_for_label = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
    _Q_for_label = nx.algorithms.community.modularity(G, _comms_for_label, weight="weight")
# Translucent module shading — convex hull around each community's nodes
from scipy.spatial import ConvexHull as _ConvexHull
import matplotlib.patches as _mpatches
_module_palette = plt.cm.tab10(np.linspace(0, 1, max(len(_comms_for_label), 1)))
for _mi, _comm in enumerate(_comms_for_label):
    _pts = np.array([pos[_n] for _n in _comm if _n in pos])
    if len(_pts) < 3:
        # Modules with 1-2 nodes can't form a hull; draw a circle around the centroid instead
        if len(_pts) >= 1:
            _c = _pts.mean(axis=0)
            _r = max(0.05, np.linalg.norm(_pts - _c, axis=1).max() * 1.5 if len(_pts) > 1 else 0.05)
            _circle = _mpatches.Circle(_c, _r, facecolor=_module_palette[_mi],
                                       edgecolor=_module_palette[_mi], linewidth=2,
                                       alpha=0.12, zorder=0)
            ax.add_patch(_circle)
        continue
    _hull = _ConvexHull(_pts)
    _hull_pts = _pts[_hull.vertices]
    _centroid = _pts.mean(axis=0)
    _inflated = _centroid + (_hull_pts - _centroid) * 1.18   # 18% outward buffer for breathing room
    _poly = _mpatches.Polygon(_inflated, closed=True,
                              facecolor=_module_palette[_mi],
                              edgecolor=_module_palette[_mi],
                              linewidth=2, alpha=0.12, zorder=0)
    ax.add_patch(_poly)

_module_summary = (
    r"$\bf{Modularity}$" + "\n"
    + f"Q = {_Q_for_label:.3f}    modules = {len(_comms_for_label)}" + "\n"
    + f"sizes: {sorted([len(c) for c in _comms_for_label], reverse=True)[:6]}"
)
ax.text(-0.10, 0.7, _module_summary,
        transform=ax.transAxes,
        fontsize=8 * (width / 10),
        ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="black",
                  alpha=0.85, linewidth=1.5))

plt.savefig(f"cooccurrence_network_p_value_FDR.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Co-occurrence network metrics: modularity + per-node connectivity ===
import networkx as nx
import pandas as pd
from json import dump
from collections import Counter

# Split into positive and negative subgraphs so co-occurrence (positive Spearman) and
# antagonistic (negative Spearman) links can be reported separately.
G_pos = G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["rho"] > 0]).copy()
G_neg = G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["rho"] < 0]).copy()

# --- 1. Modularity (Louvain) on the positive subgraph, weighted by |rho| ---
try:
    communities_pos = nx.community.louvain_communities(G_pos, weight="weight", resolution=1.0, seed=42)
    Q_pos = nx.community.modularity(G_pos, communities_pos, weight="weight")
except AttributeError:
    # Older networkx: fall back to greedy modularity
    communities_pos = list(nx.algorithms.community.greedy_modularity_communities(G_pos, weight="weight"))
    Q_pos = nx.algorithms.community.modularity(G_pos, communities_pos, weight="weight")

node_to_module = {n: i for i, comm in enumerate(communities_pos) for n in comm}
print("=== Modularity (positive-edge subgraph, |rho|-weighted) ===")
print(f"  Q = {Q_pos:.3f}")
print(f"  {len(communities_pos)} modules; sizes (desc): {sorted([len(c) for c in communities_pos], reverse=True)}")

# Same on full signed graph for comparison (some workflows treat signed weights)
try:
    communities_all = nx.community.louvain_communities(G, weight="weight", resolution=1.0, seed=42)
    Q_all = nx.community.modularity(G, communities_all, weight="weight")
    print(f"  (full graph |rho|-weighted: Q = {Q_all:.3f}, {len(communities_all)} modules)")
except Exception:
    Q_all = float("nan"); communities_all = []

# --- 2. Per-node connectivity table ---
rows = []
for n in G.nodes():
    pos_neigh = sorted([(v, G[n][v]["rho"]) for v in G.neighbors(n) if G[n][v]["rho"] > 0],
                      key=lambda x: -x[1])
    neg_neigh = sorted([(v, G[n][v]["rho"]) for v in G.neighbors(n) if G[n][v]["rho"] < 0],
                      key=lambda x: x[1])
    strength = sum(abs(G[n][v]["rho"]) for v in G.neighbors(n))
    rows.append({
        "node": n,
        "degree": G.degree(n),
        "pos_degree": len(pos_neigh),
        "neg_degree": len(neg_neigh),
        "strength_abs_rho": strength,
        "community": node_to_module.get(n, -1),
        "phylum": iterativeID_level.get(n, "Unknown"),
        "top5_pos_neighbors": "; ".join(f"{v} ({rho:+.2f})" for v, rho in pos_neigh[:5]),
        "top5_neg_neighbors": "; ".join(f"{v} ({rho:+.2f})" for v, rho in neg_neigh[:5]),
    })
metrics_df = pd.DataFrame(rows).sort_values("degree", ascending=False).reset_index(drop=True)
metrics_df.to_csv("network_metrics.csv", index=False)
print(f"\n=== Per-node connectivity (top 10 by degree) ===")
print(metrics_df[["node","degree","pos_degree","neg_degree","strength_abs_rho","community","phylum"]].head(10).to_string(index=False))

# Degree distribution snapshot
deg = metrics_df["degree"]
print(f"\nDegree distribution: n={len(deg)}  mean={deg.mean():.1f}  median={int(deg.median())}  min={int(deg.min())}  max={int(deg.max())}")
print(f"Top phyla among hubs (top decile by degree): {Counter(metrics_df.head(max(1, len(metrics_df)//10))['phylum']).most_common(5)}")

# --- 3. Per-module composition snapshot ---
print(f"\n=== Module composition (positive subgraph) ===")
for i, comm in enumerate(sorted(communities_pos, key=len, reverse=True)):
    phyla_in_module = Counter(iterativeID_level.get(n, "Unknown") for n in comm)
    top = ", ".join(f"{p}({c})" for p, c in phyla_in_module.most_common(3))
    members = sorted(comm, key=lambda x: -G.degree(x))[:3]
    print(f"  module {i:2d}  size={len(comm):3d}  top phyla: {top}  | hubs: {members}")

# --- 4. Persist a JSON neighborhood map for downstream use ---
neighborhoods = {
    n: {
        "degree": int(G.degree(n)),
        "pos_degree": int(sum(1 for v in G.neighbors(n) if G[n][v]["rho"] > 0)),
        "neg_degree": int(sum(1 for v in G.neighbors(n) if G[n][v]["rho"] < 0)),
        "strength_abs_rho": float(sum(abs(G[n][v]["rho"]) for v in G.neighbors(n))),
        "community": int(node_to_module.get(n, -1)),
        "phylum": iterativeID_level.get(n, "Unknown"),
        "neighbors": {
            v: {
                "rho": float(G[n][v]["rho"]),
                "qvalue": float(G[n][v]["qvalue"]),
                "cooccurrence": int(G[n][v]["cooccurrence"]),
            }
            for v in G.neighbors(n)
        },
    }
    for n in G.nodes()
}
dump({
    "modularity_Q_positive": float(Q_pos),
    "modularity_Q_all": float(Q_all) if Q_all == Q_all else None,
    "n_modules_positive": len(communities_pos),
    "module_sizes_positive": sorted([len(c) for c in communities_pos], reverse=True),
    "communities_positive": [sorted(list(c)) for c in communities_pos],
    "neighborhoods": neighborhoods,
}, open("network_metrics.json", "w"), indent=2)
print(f"\nWrote network_metrics.csv ({len(metrics_df)} rows) and network_metrics.json")

# --- 5. Full edge list (long format) with node metadata for downstream analyses ---
edge_rows = []
for u, v, d in G.edges(data=True):
    edge_rows.append({
        "node_a": u,
        "node_b": v,
        "rho": float(d["rho"]),
        "abs_rho": float(abs(d["rho"])),
        "sign": "positive" if d["rho"] > 0 else "negative",
        "qvalue": float(d.get("qvalue", float("nan"))),
        "pvalue": float(d.get("pvalue", float("nan"))),
        "cooccurrence": int(d.get("cooccurrence", 0)),
        "community_a": int(node_to_module.get(u, -1)),
        "community_b": int(node_to_module.get(v, -1)),
        "same_community": int(node_to_module.get(u, -1) == node_to_module.get(v, -2)),
        "phylum_a": iterativeID_level.get(u, "Unknown"),
        "phylum_b": iterativeID_level.get(v, "Unknown"),
        "same_phylum": int(iterativeID_level.get(u) == iterativeID_level.get(v)),
    })
edges_df = (pd.DataFrame(edge_rows)
              .sort_values("abs_rho", ascending=False)
              .reset_index(drop=True))
edges_df.to_csv("network_edges.csv", index=False)
print(f"\nWrote network_edges.csv ({len(edges_df)} edges, sorted by |rho|)")
print(f"  positive edges: {(edges_df['rho']>0).sum()}  negative: {(edges_df['rho']<0).sum()}")
print(f"  intra-community edges: {edges_df['same_community'].sum()} ({edges_df['same_community'].mean():.0%})")
print(f"  intra-phylum  edges: {edges_df['same_phylum'].sum()} ({edges_df['same_phylum'].mean():.0%})")
